## **Chemical Equilibrium: the $\rm{N_2O_4(g)} \rightleftharpoons 2 \; \rm{NO_2(g)}$ reaction**

<div align="left">
  <table border="1" cellpadding="6" cellspacing="0">
    <tr>
      <td bgcolor="#444444">
        <font color="#ffeb3b"><tt><b>Last updated (YYYY-MM-DD): 2026-04-12</b></tt></font>
      </td>
    </tr>
  </table>
</div>



In this notebook, we focus on a single, well-defined chemical system—the thermal dissociation of dinitrogen tetroxide into nitrogen dioxide—as a concrete example to explore the fundamental principles governing **chemical equilibrium**:

$$\rm N_2O_4 (g) \rightleftharpoons 2\,NO_2 (g)$$

This activity integrates the analysis of this equilibrium into a unified computational workflow. In particular, the equilibrium will be examined from three complementary perspectives: **classical thermodynamics**, which provides the macroscopic criteria for equilibrium through the minimization of thermodynamic potentials; **statistical thermodynamics**, which connects equilibrium properties with molecular-level information such as energies and partition functions; and **chemical kinetics**, which describes the time-dependent evolution of the reacting system and the dynamic approach to equilibrium.

The notebook is designed to guide the student through the quantitative description of equilibrium by examining how thermodynamic potentials vary with the extent of reaction, how equilibrium compositions emerge from minimization criteria under specified constraints, and how changes in temperature or external conditions influence the position of equilibrium. By working with a specific and chemically meaningful system, the notebook aims to provide an intuitive and computationally accessible introduction to equilibrium as a dynamic and quantitatively predictable state of matter, integrating thermodynamic, molecular, and kinetic viewpoints within a single computational framework.


### **Preparing the Computational Environment**  


*The next four cells prepare the environment by installing the necessary packages, importing libraries, and loading constants and functions. Make sure to run hem sequentially before doing anything else.*

⏳ **Note:** This may take **a few minutes** to complete. ⏳  

In [ ]:
#@title <small> 💻 Install Libraries (this may take a while) <small> { display-mode: "form" }
%%capture captured_output

# --- system (for LaTeX text in matplotlib) --- #
! sudo apt update -y
! sudo apt install -y cm-super dvipng texlive-latex-extra texlive-latex-recommended

# --- Python: install and update core scientific libraries --- #

# versions that are known to work well together will be selected to avoid compatibility issues
%pip install "numpy==2.0.2"
%pip install "scipy==1.16.3"

# install pyscf: as pyscf[geomopt,dispersion] gives problems, they are installed separately
%pip install --prefer-binary "pyscf==2.11.0"
%pip install "geometric==1.1"
%pip install "pyscf-dispersion==1.3.0"

%pip install "py3Dmol==2.5.3"
%pip install "pubchempy==1.0.5"
%pip install "rdkit==2025.9.1"

# !pip index versions pyscf      # check available versions
# %pip install "pyberny==0.6.3"  # install specific version
# !pip show pyberny              # check installed version

In [ ]:
#@title <small> 💻 Import Libraries and Load Constants<small> { display-mode: "form" }

# =====    Import libraries of interest    ===== #
# ---------------------------------------------- #
import os                                        #
import time                                      #
# ---------------------------------------------- #
import numpy                as np                #
#print(np.__version__)                           #
# ---------------------------------------------- #
import matplotlib.pyplot    as plt               #
import matplotlib.ticker    as mticker           #
import plotly.graph_objects as go                #
from   plotly.subplots      import make_subplots #
# ---------------------------------------------- #
from   scipy.optimize  import minimize_scalar    #
from   scipy.optimize  import root_scalar        #
# from   scipy.optimize  import minimize           #
# from   scipy.optimize  import brentq             #
# ---------------------------------------------- #
import pubchempy       as     pcp                # to retrieve from PubChem
# ---------------------------------------------- #
import rdkit                                     # SMILES --> coordinates
from   rdkit           import RDLogger           #
from   rdkit           import Chem               #
from   rdkit.Chem      import AllChem            #
# ---------------------------------------------- #
import py3Dmol                                   # 3D visualization of molecules
# ---------------------------------------------- #
import pyscf                                     # for elec struct calculations
from   pyscf.geomopt   import geometric_solver   #
from   pyscf.hessian   import thermo             #
# ---------------------------------------------- #
import ipywidgets      as     w                  # to add buttons
# ---------------------------------------------- #
from   google.colab    import files              # to access to generated files
from   google.colab    import output             #
from   IPython.utils   import io                 # to capture output
from   IPython.display import HTML               # needed for 3D visualization
from   IPython.display import display            # needed for 3D visualization
from   IPython.display import Markdown           # needed for 3D visualization
# ============================================== #

# ---- Define some constants of interest (SI) ----
m_u    = 1.66053886e-27      # atomic mass constant in kg
m_e    = 9.1093837E-31       # mass of electron
q_e    = 1.60217663E-19      # charge of electron
h      = 6.6260693E-34       # Planck's constant in J*s
k_B    = 1.3806505E-23       # Boltzmann's constant in J/K
c_0    = 2.99792558E8        # Speed of light in m/s
eps0   = 8.85418782E-12      # Vacuum permittivity
NA     = 6.02214076E23       # Avogadro's number
# ------------------------------------------------
P_o    = 1E5                 # standard pressure (P^o = 1E5 Pa = 1 bar)
c_o    = 1E3                 # 1 mol/L = 1E3 mol/m^3
R      = NA * k_B            # gas constant J/(K mol)
hbar   = h/(2*np.pi)
# ----- Universal constants to atomic units  -----
a_0    = (4*np.pi*eps0*hbar**2) / (m_e * q_e**2)
Eh     = q_e**2 / (4*np.pi * eps0 * a_0)
Hz_au  = Eh/hbar
# ------------------------------------------------
ZERO1  = 1e-300  # to avoid problems when n=0 in n*log(n)
ZERO2  = 1E-014  # if ni < ZERO2 --> ni = 0 (for chemical kinetics)
ZERO3  = 1E-010  # comparison of rotational constants (if equal --> linear)
ZERO4  = 1E-007  # for intercept method; if y=0 --> y=ZERO4 (so we can obtain the derivative)
# ------------------------------------------------
last_fig  = None
last_info = None
# ------------------------------------------------
FONTSIZE  = [11,12,14,15,16,20]
# ------------------------------------------------
NPOINTSXI = 125   # number of xi points (kinetics)
REL_XI_EQ = 0.999 # plot until REL_XI_EQ * xieq
# ------------------------------------------------

In [ ]:
#@title <small> 💻 Load General Auxiliary Functions <small> { display-mode: "form" }

# ============================================= #
def reaction_to_string(nus,molecules):
    stringR = []
    stringP = []
    for nu, molecule in zip(nus,molecules):
        if nu < 0: stringR += ["%s * %s(g)"%(-nu,molecule)]
        if nu > 0: stringP += ["%s * %s(g)"%(+nu,molecule)]
    return " + ".join(stringR) + " ⇌ " + " + ".join(stringP)
# --------------------------------------------- #
def prepare_variables(T, P, nus, n_0, xi):
    T   = np.asarray(T)
    P   = np.asarray(P)
    nus = np.asarray(nus) #, dtype=float)  # (S,)
    n_0 = np.asarray(n_0) #, dtype=float)  # (S,)
    xi  = np.asarray(xi)
    # Expand for broadcasting with xi (which can be a scalar or have shape (M, N))
    # If xi.ndim == 0 (scalar), this keeps the shape (S,) -> (S,)
    # If xi.ndim == 2, it becomes (S, 1, 1) and broadcasts to (S, M, N)
    expand = (None,)*xi.ndim
    n_0_e  = n_0[(...,) + expand]
    nus_e  = nus[(...,) + expand]
    # total sums (scalars)
    dnu    = nus.sum()
    ntot_0 = n_0.sum()
    return T, P, nus_e, n_0_e, xi, dnu, ntot_0
# ============================================= #

# ============================================= #
#        Functions for downloading FILES        #
# ============================================= #
# ---- button to download file ----
def download_file(fname):
    btn = w.Button(
            description=f"Download {fname}",
            icon="download",
            button_style="primary",
            layout=w.Layout(width='250px', height='38px', margin='0 0 0 55px')
    )
    # action when clicking
    btn.on_click(lambda _: files.download(fname))
    return btn
# --------------------------------------------- #
def _on_download_clicked(_,_case_):
    if _case_.startswith("kin"): init_conditions = f"{T_slider.value:.0f}K_{P_slider.value:.2f}bar_{V_slider.value:.2f}L_{yA_slider.value:.2f}"

    # --- get figure from global variable ---
    fig = last_fig
    if fig is None:
        print("No figure yet; move a slider to generate the plot...")
        return

    # --- get file name ---
    if   _case_ == "PT"       : fname = f"equilibrium_T{T_slider.value:.0f}K_p{P_slider.value:.2f}bar.svg"
    elif _case_ == "VT"       : fname = f"equilibrium_T{T_slider.value:.0f}K_V{V_slider.value:.2f}L.svg"
    elif _case_ == "intercept": fname = f"intercept_T{T_slider.value:.0f}K_p{P_slider.value:.2f}bar.svg"
    elif _case_ == "dof"      : fname = "vib_dof.svg"
    elif _case_ == "kinPT"    : fname = f"chemkinPT_{init_conditions:s}.svg"
    elif _case_ == "kinVT"    : fname = f"chemkinVT_{init_conditions:s}.svg"
    else                      : raise Exception

    if _case_.startswith("kin"): original_size = fig.get_size_inches()
    if _case_.startswith("kin"): fig.set_size_inches(10,8)
    fig.savefig(fname, bbox_inches='tight',dpi=300)
    if _case_.startswith("kin"): fig.set_size_inches(original_size)
    files.download(fname)
# --------------------------------------------- #
def pyscf_download(molecule,functional,basis,which_ones=[]):

    xyz_guess,xyz_opt,output_opt,output_frq = files_of_interest(molecule,functional,basis)
    print(rf"     - file(s) to download:")
    if 1 in which_ones:
         print(rf"       {xyz_opt:s}")
         btn1 = download_file(xyz_opt)   ; display(btn1)
    if 2 in which_ones:
         print(rf"       {output_opt:s}")
         btn2 = download_file(output_opt); display(btn2)
    if 3 in which_ones:
         print(rf"       {output_frq:s}")
         btn3 = download_file(output_frq); display(btn3)
# ============================================= #

# ============================================= #
# Functions for part 1: CHEMICAL THERMODYNAMICS #
# ============================================= #
# In following functions, it is assumed that    #
# all data are provided in SI units.            #
# Moreover, P is used for total pressure; lower #
# p is used for partial pressures.              #
# ============================================= #
def limits_xi(n_0,nus):
    # maximum value of xi (calculated considering consumption of reactants)
    xi_max = min([-n_0_i/nu_i for n_0_i,nu_i in zip(n_0,nus) if nu_i < 0])
    # minimum value of xi (calculated considering consumption of products)
    xi_min = max([-n_0_i/nu_i for n_0_i,nu_i in zip(n_0,nus) if nu_i > 0])
    # Ensure numerical zeros are displayed as +0.0 for clarity
    if xi_min == -0.0: xi_min = 0.0
    if xi_max == -0.0: xi_max = 0.0
    return xi_min,xi_max
# --------------------------------------------- #
def get_DHo(T,refdata):
    '''Delta{H}^o as a function of T'''
    DHo_ref,DSo_ref,DCPo_ref,T_ref = refdata
    DHo_T = DHo_ref + DCPo_ref * T * (1 - T_ref/T)
    return DHo_T
# --------------------------------------------- #
def get_DGo(T,refdata):
    '''Delta{G}^o as a function of T'''
    DHo_ref,DSo_ref,DCPo_ref,T_ref = refdata
    DGo_T  = DHo_ref - T * DSo_ref
    DGo_T += DCPo_ref  * ( T - T_ref + T*np.log(T_ref / T))
    return DGo_T
# --------------------------------------------- #
def get_Gast(T,P,nus,refdata):
    DGast = get_DGo(T,refdata) + R*T*np.log(P/P_o) * nus.sum()
    return DGast
# --------------------------------------------- #
def get_DDGmix(xi, T, P, n_0, nus):
    """
    Accepts xi as either a scalar or an array (e.g., a mesh of shape (M, N)); T and p can be scalars or broadcast-compatible arrays.
    """

    T, P, nus, n_0, xi, dnu, ntot_0 = prepare_variables(T, P, nus, n_0, xi)

    # magnitudes dependent on xi
    n_xi    = n_0    + xi * nus   # (S,) o (S,M,N)
    ntot_xi = ntot_0 + xi * dnu   # scalar or (M,N)

    # Get molar fractions
    y_xi = n_xi /ntot_xi
    y_0  = n_0  /ntot_0

    # Notice that 0*ln(0) --> 0
    maskA = (n_xi > ZERO1) & (y_xi > ZERO1)
    termA = np.zeros_like(y_xi, dtype=float)
    termA[maskA] = n_xi[maskA] * np.log(y_xi[maskA])

    maskB = (n_0  > ZERO1) & (y_0  > ZERO1)
    termB = np.zeros_like(y_0, dtype=float)
    termB[maskB] = n_0[maskB]  * np.log(y_0[maskB])

    DDGmix = R*T*np.sum(termA-termB, axis=0)
    return DDGmix
# --------------------------------------------- #
def get_G_PT(xis,P,T,n_0,nus,refdata):
    '''Gibbs free energy for a given T,p'''
    Gast   = get_Gast(T,P,nus,refdata)
    DDGmix = get_DDGmix(xis,T,P,n_0,nus)
    DGtot  = Gast * xis + DDGmix
    return DGtot
# --------------------------------------------- #
def get_A_VT(xis,V,T,n_0,nus,refdata):
    '''Helmholtz free energy for a given T,V'''
    ntot_0  = n_0.sum()
    ntot_xi = ntot_0 + nus.sum() * xis
    DGo_T   = get_DGo(T,refdata)
    P_xi    = ntot_xi*R*T/V
    term_lineal = DGo_T * xis
    termA   = ntot_xi * R *T * (np.log( P_xi/P_o                  )-1)
    termA  -= ntot_0  * R *T * (np.log((P_xi/P_o)*(ntot_0/ntot_xi))-1)
    DDGmix  = get_DDGmix(xis,T,P_xi,n_0,nus)
    term_nolin  = termA+DDGmix
    DAtot   = term_lineal + term_nolin
    return DAtot, term_lineal, term_nolin, P_xi
# --------------------------------------------- #
def get_Qp_PT(n_0,nus,xi,P):
    '''Quotient of reaction (pressure)'''
    # Get partial pressures for this value of xi
    n_xi = n_0 + xi*nus
    y_xi = n_xi/n_xi.sum()
    p_xi = y_xi * P
    with np.errstate(divide='ignore', invalid='ignore', over='ignore', under='ignore'):
         Qp = np.prod([(p_xi_j/P_o)**nu_j for nu_j,p_xi_j in zip(nus,p_xi)])
    return Qp
# --------------------------------------------- #
def get_Qp_VT(n_0,nus,xi,V,T):
    '''Quotient of reaction (volume)'''
    n_xi = n_0 + xi*nus
    y_xi = n_xi/(n_xi.sum())
    P    = n_xi.sum() *R*T/V
    p_xi = y_xi * P
    with np.errstate(divide='ignore', invalid='ignore', over='ignore', under='ignore'):
         Qp = np.prod([(p_xi_j/P_o)**nu_j for nu_j,p_xi_j in zip(nus,p_xi)])
    return Qp
# --------------------------------------------- #
def get_xieq_PT(P,T,n_0,nus,refdata):
    '''Extent of reaction in terms of P and T'''
    # Get Eq constant
    dGo_T = get_DGo(T,refdata)
    Kp_T  = np.exp(-dGo_T/R/T)
    # Get limits for xi
    xi_min,xi_max = limits_xi(n_0,nus)
    xi_guess      = 0.5 * (xi_max + xi_min)
    result        = root_scalar(lambda xi: get_Qp_PT(n_0,nus,xi,P)-Kp_T,x0=xi_guess,bracket=(xi_min,xi_max),xtol=1E-8)
    xi_eq         = float(result.root)
    return xi_eq
# --------------------------------------------- #
def get_xieq_VT(V,T,n_0,nus,refdata):
    '''Extent of reaction in terms of V and T'''
    # Get Eq constant
    dGo_T = get_DGo(T,refdata)
    Kp_T  = np.exp(-dGo_T/R/T)
    # Get limits for xi
    xi_min,xi_max = limits_xi(n_0,nus)
    xi_guess      = 0.5 * (xi_max + xi_min)
    result        = root_scalar(lambda xi: get_Qp_VT(n_0,nus,xi,V,T)-Kp_T,x0=xi_guess,bracket=(xi_min,xi_max))
    xi_eq         = float(result.root)
    return xi_eq
# ============================================= #

# ============================================= #
# Functions for part 1: STATIST.-THERMODYNAMICS #
# ============================================= #
def level_to_string(functional,basis):
    level      = rf"{functional.upper():s}_{basis.upper():s}"
    # replace * by _ast_ to avoid name problems
    level      = level.replace("**","_ast_ast_")
    level      = level.replace("*","_ast_")
    return level
# --------------------------------------------- #
def files_of_interest(molecule,functional="",basis=""):

    level      = level_to_string(functional,basis)
    # filenames
    try   : sgrid = rf".grid{DFTGRID:d}"
    except: sgrid = rf""
    xyz_guess  = rf"xyz_guess-{molecule:s}.xyz"
    xyz_opt    = rf"xyz_optim-{molecule:s}.{level:s}{sgrid:s}.xyz"
    output_opt = rf"pyscf_opt-{molecule:s}.{level:s}{sgrid:s}.out"
    output_frq = rf"pyscf_frq-{molecule:s}.{level:s}{sgrid:s}.out"
    # return filenames
    return xyz_guess,xyz_opt,output_opt,output_frq
# --------------------------------------------- #
def pubchem_cid(cid):
    '''PUBCHEM: get geom'''
    symbols,coords,smiles = None,None,None

    try   : compound = pcp.Compound.from_cid(int(cid))
    except: compound = None

    if compound is not None:
       print(rf"     - geometry retrieved!")
       print(rf"")
       print(rf"     - information:")
       print(rf"       * PubChem CID       : {compound.cid}")
       print(rf"       * molecular formula : {compound.molecular_formula:s}")
       # print(rf"       * charge            : {compound.charge:d}")
       print(rf"       * SMILES            : '{compound.smiles:s}'")
       print("")
       smiles  = compound.smiles
       # for diatomic molecules, 3D geoemtry is not stored in PubChem
       try   : geom = pcp.get_compounds(int(cid),"cid",record_type='3d')[0]
       except: geom = pcp.get_compounds(int(cid),"cid")[0]
       symbols = [atom.element           for atom in geom.atoms]
       # z-coordinate may be None (when record_type='3d' is not used)
       coords  = [tuple(0.0 if c is None else c for c in (atom.x, atom.y, atom.z)) for atom in geom.atoms]
    return symbols,coords,smiles
# --------------------------------------------- #
def rdkit_smiles2geom(smiles):
    '''RDKIT: get geom (from SMILES)'''

    symbols,coords = None,None
    try:
       m   = Chem.AddHs(Chem.MolFromSmiles(smiles))
       cid = AllChem.EmbedMolecule(m)
       if cid >= 0:
          symbols = [atom.GetSymbol() for atom in m.GetAtoms()]
          coords  = [list(m.GetConformer().GetAtomPosition(i)) for i in range(m.GetNumAtoms())]
          mformu  = {s:0 for s in symbols}
          for s in symbols: mformu[s] += 1
          mformu  = "".join([k+str(v) if v!=1 else k for k,v in mformu.items()])
          print(rf"     - geometry generated!")
          print(rf"")
          print(rf"     - information:")
          print(rf"       * molecular formula = {mformu:s}")
          print(rf"       * SMILES            = '{smiles:s}'")
          print("")
    except: pass

    return symbols,coords,smiles
# --------------------------------------------- #
def data_2_xyz(symbols,xcc,fname,smiles=""):
    '''data (from smiles) to .xyz file'''
    string  = rf"{len(symbols):.0f}"+"\n"
    string += rf"Cartesian coordinates for SMILES: {smiles:s}"+"\n"
    for idx,symbol in enumerate(symbols):
        xx,yy,zz = [coord for coord in xcc[idx]]
        string += rf"{symbol:2s}     {xx:9.5f}  {yy:9.5f}  {zz:9.5f}" + "\n"
    with open(fname,'w') as asdf: asdf.write(string)
    # download file button
    btn = download_file(fname)
    display(btn)
    print("")
# --------------------------------------------- #
def read_xyz(filename):
    with open(filename) as f: lines = f.readlines()
    nat = int(lines[0])
    symbols = []
    coords = []
    for line in lines[2:2+nat]:
        parts = line.split()
        if len(parts) < 4: continue
        sym, x, y, z = parts[:4]
        symbols.append(sym)
        coords.append([float(x), float(y), float(z)])
    return symbols, np.array(coords)
# --------------------------------------------- #
def create_visualization_xyz(xyz_file):
    # draw molecule
    view = py3Dmol.view(width=300, height=200)
    view.addModel(open(xyz_file, 'r').read(), 'xyz')
    view.setStyle({'stick': {'singleBonds': False}, 'sphere': {'scale': 0.3}})
    # Automatic labels: 0-based
    index_style = {"fontColor": "black","fontSize": 12,"showBackground": True ,"backgroundColor": "white","backgroundOpacity": 0.7}
    #elemn_style = {"fontColor": "black","fontSize": 12,"showBackground": False,"inFront": True,"alignment": "center","offset": {"x": 0, "y": 10, "z": 0}}
    view.addPropertyLabels("index",{},index_style)
    #view.addPropertyLabels("elem" ,{},elemn_style)
    #zoom
    view.zoomTo()
    view.zoom(2.5)
    return view
# --------------------------------------------- #
def pyscf_carryout_opt(molecule,unpaired,charge,functional,basis,bsym=False):

    # Files of interest
    xyz_guess,xyz_opt,output_opt,output_frq = files_of_interest(molecule,functional,basis)

    # Generate molecule from .xyz file
    with io.capture_output() as captured:
        mol = pyscf.gto.Mole()
        mol.atom         = xyz_guess
        mol.spin         = unpaired
        mol.charge       = charge
        mol.basis        = basis
        mol.output       = output_opt
        mol.verbose      = 4
        if bsym:
           mol.symmetry     = True
           mol.symmetrize   = True
           mol.symmetry_tol = 1e-2
        mol.build()

    # Define DFT method and mesh grid (from 0 to 9, default = 3)
    if unpaired == 0: mf = mol.RKS(xc=functional)
    else            : mf = mol.UKS(xc=functional)
    mf.grids.level = DFTGRID
    mf.max_cycle   = 200
    mf.conv_tol    = 1e-7

    # run SCF and avoid printing information
    print("     - single point calculation of guess geometry...")
    t1   = time.time()
    with io.capture_output() as captured: mf   = mf.run()
    Etot = mf.e_tot
    t2   = time.time()
    print(rf"       Etot(guess geometry) = {Etot:.5f} hartree")
    print(rf"       SCF took {t2-t1:.1f} seconds")

    # optimization [avoid printing information]
    t1   = time.time()
    print("     - geometry optimization...")
    conv_params = {}
    conv_params['convergence_energy'] = 5.0e-7  # Eh
    conv_params['convergence_gmax'  ] = 2.0e-4  # Eh/Bohr
    with io.capture_output() as captured:
       try:
          opt_geom = geometric_solver.optimize(mf, maxsteps=300, **conv_params)
       except:
          conv_params['convergence_energy'] = 1.0e-6  # Eh
          conv_params['convergence_gmax'  ] = 1.0e-4  # Eh/Bohr
          opt_geom = geometric_solver.optimize(mf, maxsteps=300, **conv_params)
    for line in opt_geom.tostring().split("\n"): print(7*" "+line)
    t2   = time.time()
    print(rf"       geometry opt. took {t2-t1:.1f} seconds")

    pyscf.gto.tofile(opt_geom,xyz_opt)
# --------------------------------------------- #
def pyscf_carryout_frq(molecule,unpaired,charge,functional,basis,bsym=False):

    # Files of interest
    xyz_guess,xyz_opt,output_opt,output_frq = files_of_interest(molecule,functional,basis)

    # Generate molecule from optimized geometry stored in xyz file
    with io.capture_output() as captured:
        mol = pyscf.gto.Mole()
        mol.atom         = xyz_opt
        mol.spin         = unpaired
        mol.charge       = charge
        mol.basis        = basis
        mol.output       = output_frq
        mol.verbose      = 6
        if bsym:
           mol.symmetry     = True
           mol.symmetrize   = True
           mol.symmetry_tol = 1e-2
        mol.build()

    # Define DFT method and mesh grid
    if unpaired == 0: mf = mol.RKS(xc=functional)
    else            : mf = mol.UKS(xc=functional)
    mf.grids.level = DFTGRID
    mf.max_cycle   = 200
    mf.conv_tol    = 1e-7

    print("     - Hessian calculation...")
    t1   = time.time()

    # run SCF and avoid printing information
    with io.capture_output() as captured: mf   = mf.run()
    Etot = mf.e_tot
    print(rf"       Etot(optim geometry) = {Etot:.5f} hartree")

    # Carry out Hessian calculation
    with io.capture_output() as captured: hessian = mf.Hessian().kernel()

    # add Hessian matrix to output_frq
    H4 = np.asarray(hessian)
    with open(output_frq, "a") as f:
        f.write("\n*** HESSIAN BY 3x3 ATOMIC BLOCKS (Hartree/Bohr^2) ***\n")
        for i in range(mol.natm):
            for j in range(mol.natm):
                f.write(f"\n# Block ({i+1},{j+1})  [atom {i+1} vs atom {j+1}]\n")
                np.savetxt(f, H4[i, j], fmt=" % .6e")

    t2   = time.time()
    print(rf"       Hessian calc. took {t2-t1:.1f} seconds")

    # return data
    return mol, mf, hessian
# --------------------------------------------- #
def pyscf_extract(mol, mf, hessian, unpaired):

    # translational info
    masses  = mol.atom_mass_list(isotope_avg=True)
    mass    = sum(masses)

    # vibrational info
    freqs       = thermo.harmonic_analysis(mf.mol, hessian)['freq_au']
    au2hz       = (1/2/np.pi)*(Eh/m_u/a_0**2)**0.5
    freqs_Hz    = [f*au2hz for f in freqs]
    wavenum_m   = [f/c_0   for f in freqs_Hz]

    info_thermo = thermo.thermo(mf, freqs) # by default, at 298.15 K and 101325 Pa
    ZPE         = info_thermo['ZPE'][0]

    # rotational info
    A,B,C = thermo.rotation_const(masses,mol.atom_coords(),unit='GHz')
    # if linear, two are equal, the other is infinity
    if   np.isinf(A) and abs(B-C) < ZERO3: A,B,C,linear = B,None,None,True
    elif np.isinf(B) and abs(A-C) < ZERO3: A,B,C,linear = C,None,None,True
    elif np.isinf(C) and abs(A-B) < ZERO3: A,B,C,linear = A,None,None,True
    else                                 :       linear =             False
    sigma = thermo.rotational_symmetry_number(mol)

    # electronic info
    E0 = info_thermo['E0' ][0]

    # collect data
    data = {}
    data["natoms"  ] = mol.natm
    data["mass"    ] = mass
    data["rotcons" ] = [i*1E9 if i is not None else i for i in [A,B,C]] # in Hz
    data["rotsigma"] = sigma
    data["islinear"] = linear
    data["freqs"   ] = wavenum_m # in 1/m
    data["ZPE"     ] = ZPE       # in hartree
    data["unpaired"] = unpaired
    data["E0"      ] = E0        # in hartree

    # return data
    return data
# --------------------------------------------- #
def optimize_and_freqs(molecule,unpaired,charge,functional,basis,bsym=False):
    # files
    xyz_guess,xyz_opt,output_opt,output_frq = files_of_interest(molecule,functional,basis)
    # check if calculation was already carried out
    args = (molecule,unpaired,charge,functional,basis,bsym)
    # geometry optimization
    if not os.path.exists(xyz_opt): pyscf_carryout_opt(*args)
    # Hessian calculation
    mol, mf, hessian = pyscf_carryout_frq(*args)
    # Extract data
    return pyscf_extract(mol,mf,hessian,unpaired)
# --------------------------------------------- #
def pfn_translational(T,mass):

    beta    = 1/(k_B*T)

    # to SI
    mass_SI = mass * m_u

    # translational contribution
    V_per_molec = 1 / (beta * P_o)
    broglie_wvl = ((beta * h**2) / (2 * np.pi * mass_SI))**(0.5)
    q_translat  = V_per_molec / broglie_wvl**3

    # d(lnq_tr)/dbeta at constant volume
    dlnqdbeta_v = - 3/2 * (1/beta)

    # d(lnq_tr)/dbeta at constant pressure
    dlnqdbeta_p = - 5/2 * (1/beta)

    return q_translat, dlnqdbeta_v
# --------------------------------------------- #
def pfn_rotational(T,A,B,C,linear,sigma):

    beta = 1/(k_B*T)

    # rotational constants (Hz --> 1/m) and rot temperature
    A /= c_0; theta_A = (h*c_0/k_B) * A

    if linear:
      q_rotational = T / theta_A
      dlnqdbeta    = - (1/beta)
    else:
      B /= c_0; theta_B = (h*c_0/k_B) * B
      C /= c_0; theta_C = (h*c_0/k_B) * C
      q_rotational = (np.pi * T**3 / (theta_A * theta_B * theta_C))**(1/2)
      dlnqdbeta    = - 3/2 * (1/beta)

    q_rotational /= sigma

    return q_rotational, dlnqdbeta
# --------------------------------------------- #
def pfn_vibrational(T,freqs):

    beta = 1/(k_B*T)

    q_vibrational = 1.0
    dlnqdbeta     = 0.0
    for freq in freqs:
        nu    = freq * c_0 # in Hz
        theta = h*nu/k_B
        q_vibrational *= 1 / (1 - np.exp(-theta/T)) # from ZPE
        dlnqdbeta     += - h*nu / (np.exp(theta/T) - 1)

    return q_vibrational, dlnqdbeta
# --------------------------------------------- #
def pfn_electronic(T,unpaired):

    q_electronic = unpaired + 1
    dlnqdbeta    = 0.0

    return q_electronic, dlnqdbeta
# --------------------------------------------- #
def compute_thermodynamics(T,molecule,key,dftdata):

    # unpack data
    natoms   = dftdata[molecule][key]["natoms"  ]
    mass     = dftdata[molecule][key]["mass"    ]
    A,B,C    = dftdata[molecule][key]["rotcons" ]
    linear   = dftdata[molecule][key]["islinear"]
    sigma    = dftdata[molecule][key]["rotsigma"]
    freqs    = dftdata[molecule][key]["freqs"   ]
    unpaired = dftdata[molecule][key]["unpaired"]
    E0       = dftdata[molecule][key]["E0"      ]
    ZPE      = dftdata[molecule][key]["ZPE"     ]

    # Get partition functions
    q_translat   , dlnqtdbeta = pfn_translational(T,mass)
    if natoms == 1:
       q_rotational , dlnqrdbeta = 1.0, 0.0
       q_vibrational, dlnqvdbeta = 1.0, 0.0
    else:
       q_rotational , dlnqrdbeta = pfn_rotational(T,A,B,C,linear,sigma)
       q_vibrational, dlnqvdbeta = pfn_vibrational(T,freqs)
    q_electronic , dlnqedbeta = pfn_electronic(T,unpaired)
    q_tot         = q_translat * q_rotational * q_vibrational * q_electronic
    dlnqdbeta_tot = dlnqtdbeta + dlnqrdbeta + dlnqvdbeta + dlnqedbeta

    # Info line
    line  = rf" {q_translat:.4E} | {q_rotational:.4E} | {q_vibrational:.4E} | {q_electronic:.4E} | {q_tot:.4E} | {(E0+ZPE):12.5f}"

    # calculate U and S (in S.I.; per molecule)
    Eref = (E0+ZPE)*Eh
    U    = Eref - dlnqdbeta_tot
    S    = - dlnqdbeta_tot/T + k_B * np.log(q_tot * np.e)

    # calculate H and G (per molecule)
    H    = U + k_B * T
    G    = H - S   * T

    # return data in J and J/K
    return U, H, S, G, line
# --------------------------------------------- #
def sum_squared_errors(DCP,T,DGo_T,T_ref,DHo_ref,DSo_ref):
    refdata   = (DHo_ref,DSo_ref,DCP*R,T_ref)
    DGo_model = get_DGo(T,refdata)
    residuals = DGo_model - DGo_T
    return np.sum(residuals**2)
# --------------------------------------------- #
def calculate_rms(x1,x2):
    return (sum([(i-j)**2 for i,j in zip(x1,x2)])/len(x1))**0.5
# --------------------------------------------- #
def freq_to_nu_and_theta(freq_cm):
    nu    = 100*freq_cm*c_0
    theta = h*nu/k_B
    return nu, theta
# --------------------------------------------- #
def vib_contribution(freq_cm,T):
    nu,theta = freq_to_nu_and_theta(freq_cm)
    thetaT   = theta/T
    contri   = (thetaT * np.exp(-thetaT/2)/(1-np.exp(-thetaT))) **2
    return contri
# --------------------------------------------- #
def vib_contri_avera(freq_cm,T1,T2):
    nu,theta = freq_to_nu_and_theta(freq_cm)
    average  = 1/(np.exp(theta/T2)-1) - 1/(np.exp(theta/T1)-1)
    average  = average * theta/(T2-T1)
    return average
# ============================================= #

# ============================================= #
#      FUNCTIONS FOR PRINTING INFORMATION       #
# ============================================= #
def print_info_eq(magnitude,PVT_0,PVT_eq,molecules,xi_eq,n_eq,y_eq,p_eq,P_eq,Kp_v1,Kp_v2,Ky_v1,Ky_v2):

    P_0 ,V_0 ,T_0  = PVT_0
    P_eq,V_eq,T_eq = PVT_eq

    sP_0  = rf"{P_0/1E5:.2f}"
    sP_eq = rf"{P_eq/1E5:.2f}"
    Pformat = max(len(sP_0),len(sP_eq))

    sV_0  = rf"{V_0*1E3:.2f}"
    sV_eq = rf"{V_eq*1E3:.2f}"
    Vformat = max(len(sV_0),len(sV_eq))

    print("")
    print(fr"Initial  conditions: ({T_0:.2f}K,{sP_0:{Pformat:d}s}bar,{sV_0:{Vformat:d}s}L)")
    print("")
    print(fr"Equilib. conditions: ({T_eq:.2f}K,{sP_eq:{Pformat:d}s}bar,{sV_eq:{Vformat:d}s}L)")
    print("")
    print(fr"   ==> equilibrium found at xi_eq = {xi_eq:.4f} mol")
    print("")

    print(fr"   Number of moles & molar fraction at equilibrium (from xi_eq):")
    for j,molecule in enumerate(molecules):
        n_eq_j = n_eq[j]
        y_eq_j = y_eq[j]
        print(fr"   * n_i = {n_eq_j:6.4f} mol, y_i = {y_eq_j:7.4f} (i = {molecule:s})")
    print("")

    print(fr"   Partial pressures at equilibrium:")
    for j,molecule in enumerate(molecules):
        p_eq_j = p_eq[j]/1E5
        print(fr"   * p_i = {p_eq_j:7.4f} bar (i = {molecule:s})")
    print("")


    if 9999 > Kp_v1 > 0.100: sformat1 = ".3f"
    else                   : sformat1 = ".2E"
    if 9999 > Ky_v1 > 0.100: sformat2 = ".3f"
    else                   : sformat2 = ".2E"

    reldiff = 100*abs(Kp_v2-Kp_v1)/Kp_v1
    if reldiff < 5.0:
        print(fr"   Value of Kp:")
        print(fr"   * from Delta_r{{G}}^o --> Kp = {Kp_v2:{sformat1}} [*]")
        print(fr"   * from p_i values   --> Kp = {Kp_v1:{sformat1}}")
        print("")
        print(fr"   Value of Ky:")
        if Ky_v2 is not None:
          print(fr"   * from Delta_r{{G}}^* --> Ky = {Ky_v2:{sformat2}} [*]")
        print(fr"   * from y_i values   --> Ky = {Ky_v1:{sformat2}}")
        print("")
        print(fr"   The previous values for the equilibrium constants may vary slightly")
        print(fr"   due to numerical errors. Trust the values with the [*].")
    else:
        print(fr"   Value of Kp:")
        print(fr"   * from Delta_r{{G}}^o --> Kp = {Kp_v2:{sformat1}}")
        print("")
        if Ky_v2 is not None:
          print(fr"   Value of Ky:")
          print(fr"   * from Delta_r{{G}}^* --> Ky = {Ky_v2:{sformat2}}")
    print("")
# --------------------------------------------- #
def geometric_info_xyz(xyz_file,geominfo):
    info = ""
    symbols, coords = read_xyz(xyz_file)
    for ii in geominfo:
        if len(ii) == 2:
          at1,at2 = ii
          distance = np.linalg.norm(coords[at1] - coords[at2])
          sbond    = rf"{symbols[at1]:s}{at1:d}-{symbols[at2]:s}{at2:d}"
          info    += rf"       * d({sbond:8s}) = {distance:.3f} Å" + "\n"
        elif len(ii) == 3:
          at1,at2,at3 = ii
          v1 = coords[at1] - coords[at2]
          v2 = coords[at3] - coords[at2]
          cosang = np.dot(v1, v2) / (np.linalg.norm(v1)*np.linalg.norm(v2))
          cosang = np.clip(cosang, -1.0, 1.0)
          angle  = np.degrees(np.arccos(cosang))
          sangle = rf"{symbols[at1]:s}{at1:d}-{symbols[at2]:s}{at2:d}-{symbols[at3]:s}{at3:d}"
          info  += rf"       * ∠({sangle:8s}) = {angle:.1f}°" + "\n"
    return info
# --------------------------------------------- #
def pyscf_printdata(data):
    '''print information of interest after Hessian calc'''
    # unpack data
    mass     = data["mass"    ]
    A,B,C    = data["rotcons" ]
    linear   = data["islinear"]
    sigma    = data["rotsigma"]
    freqs    = data["freqs"   ]
    ZPE      = data["ZPE"     ]
    unpaired = data["unpaired"]
    E0       = data["E0"      ]

    # print fata
    INFO  = rf"     - total mass (amu)              : {mass:.2f}" + "\n"
    if linear: INFO += rf"     - rotational constant  (GHz)    : {(A*1E-9):.2f}" + "\n"
    else     : INFO += rf"     - rotational constants (GHz)    : " + "  ".join(["%.2f"%(ii*1E-9) for ii in (A,B,C)]) + "\n"
    INFO += rf"     - rotational symmetry number    : {sigma:d}" + "\n"
    INFO += rf"     - vibrational wavenumbers (1/cm):"+"\n"
    # frequencies are, actually, wavenumbers in (1/m)
    for i in range(0,len(freqs),5):
        INFO += "         "+"  ".join(["%8.2f"%(ii/100) for ii in freqs[i:i+5]])+"\n"
    INFO += rf"     - zero point energy (hartree)   : {ZPE:.5f}"+"\n"
    print(INFO)
# ============================================= #


# ============================================= #
# ----          PLOTTING FUNCTIONS         ---- #
# ============================================= #
def plot_DGo_T(T,T_ref,refdata):
    # Calculation of DG at Tref
    DGo_ref = get_DGo(T_ref,refdata)
    Keq_ref = np.exp(-DGo_ref/R/T_ref)

    # Calculation of DG at other temperatures
    DGo_T = get_DGo(T,refdata)
    Keq_T = np.exp(-DGo_T/R/T)

    # Plot results
    plt.rcParams['text.usetex'] = True

    # Create figure with two panels side by side
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    # LEFT PANEL #
    labeldot = r"$\Delta_{r}{G}^\circ(T_{\rm ref}) = %.2f \;\; \mathrm{kJ/mol}$"%(DGo_ref/1000)
    ax[0].plot(T    ,DGo_T  /1000,'k-',zorder=1)
    ax[0].plot(T_ref,DGo_ref/1000,'ro',zorder=3,label=labeldot)
    ax[0].tick_params(axis='both', labelsize=FONTSIZE[4])
    ax[0].set_xlabel(r'$T \;\; (\mathrm{K})$'                        ,fontsize=FONTSIZE[5])
    ax[0].set_ylabel(r'$\Delta_{r} G^{\circ} \;\; (\mathrm{kJ/mol})$',fontsize=FONTSIZE[5])
    ax[0].legend(loc="best",fontsize=FONTSIZE[3])

    # RIGHT PANEL #
    if 1E-2 < Keq_ref < 1000: sKeq_ref = "%.3f"%Keq_ref
    else                    : sKeq_ref = "%.3e"%Keq_ref
    labeldot = r"$K_{p}^\circ(T_{\rm ref}) = %s$"%sKeq_ref
    ax[1].plot(T    ,Keq_T  ,'k-',zorder=1)
    ax[1].plot(T_ref,Keq_ref,'ro',zorder=3,label=labeldot)
    ax[1].tick_params(axis='both', labelsize=FONTSIZE[4])
    ax[1].set_xlabel(r'$T \;\; (\mathrm{K})$'     ,fontsize=FONTSIZE[5])
    ax[1].set_ylabel(r'$K_{p}^\circ(T_{\rm ref})$',fontsize=FONTSIZE[5])
    ax[1].legend(loc="best",fontsize=FONTSIZE[3])

    # ---- download button ----
    fname = 'plot_DGo_T.svg'
    fig   = plt.gcf()
    fig.savefig(fname, bbox_inches='tight')
    btn   = download_file(fname)

    # ---- show plot and close ----
    display(btn)
    plt.show()
    plt.close()

    return DGo_T
# --------------------------------------------- #
def plot_DGo_T_statmech(T,DGo_T,DGo_model,DCP,T_ref,DGo_ref,key):

    plt.rcParams['text.usetex'] = True

    if DCP is not None: DCP = DCP/R
    # Plot results
    plt.plot(T,[ii/1000 for ii in DGo_T]    ,'k-',zorder=1,label=r"Stat-Mech")

    if DGo_model is not None:
       plt.plot(T,[ii/1000 for ii in DGo_model],'r--',zorder=1,label=r"Eq. (1) with $\Delta_{r}C_{p}^\circ= %.2f\cdot R$"%DCP)

    if T_ref is not None:
       plt.plot(T_ref,DGo_ref/1000,'ro')

    # Format plot
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    plt.xlabel(r'$T \;\; (\mathrm{K})$'                        ,fontsize=16)
    plt.ylabel(r'$\Delta_{r} G^{\circ} \;\; (\mathrm{kJ/mol})$',fontsize=16)

    plt.legend(loc="best",fontsize=12)

    if DGo_model is not None:
       level = level_to_string(key[0],key[1])
       fname = rf"plot_DGo_T_stat_{level:s}_{DCP:5.2f}.svg"
       # ---- download button ----
       fig = plt.gcf()
       fig.savefig(fname, bbox_inches='tight')
       btn = download_file(fname)
       display(btn)

    # ---- show plot and close ----
    plt.show()
    plt.close()
# --------------------------------------------- #
def plot_gibbshelmholtz(T,DGo_T,refdata):
    # --- DH^o at each temperature ---
    DHo_T = get_DHo(T,refdata)
    yy    = DHo_T / (R * T**2)

    # Plot results
    fig, ax = plt.subplots()
    ax.plot(T,yy,'k-',zorder=1,label=r'$(a) \;\; f=\Delta_r H^{\circ}/T^2$ \quad\quad\quad\quad [analytic]')

    # Format plot
    ax.tick_params(axis='x', labelsize=FONTSIZE[4])
    ax.tick_params(axis='y', labelsize=FONTSIZE[4])

    ax.set_xlabel(r'$T \;\; (\mathrm{K})$'       , fontsize=FONTSIZE[5])
    ax.set_ylabel(r'$f/R \;\; (\mathrm{K}^{-1})$', fontsize=FONTSIZE[5])

    # --- Calculate numerical derivative ---
    numslope = np.gradient(-DGo_T / (R*T) , T)

    # --- Add data to plot (removing first and last points, due to numerical error) ---
    ax.plot(T[1:-1],numslope[1:-1],'xr',zorder=2, markeredgewidth=2.5,label=r'$(b)  \;\; f = -d/dT(\Delta_r G^{\circ}/T)$ \;\, [numeric]')

    ax.legend(loc="best",fontsize=FONTSIZE[3])

    # ---- download button ----
    fname = "plot_gibbshelmholtz.svg"
    fig   = plt.gcf()
    fig.savefig(fname, bbox_inches='tight')
    btn   = download_file(fname)

    # Show plot, button and close
    display(btn)
    display(fig)
    plt.close()
# --------------------------------------------- #
def plot_DG_PT(T,P, fixed_args):

    molecules,nus,n_0,xis,refdata = fixed_args

    V_0 = n_0.sum()*R*T/P

    # ---- Calculate DG* ----
    DGo   = get_DGo(T,refdata)
    Gast  = get_Gast(T,P,nus,refdata)

    # ---- Terms in DGtot ----
    DGlin  = Gast * xis
    DDGmix = get_DDGmix(xis,T,P,n_0,nus)
    DGtot  = DGlin + DDGmix

    # ---- minimum of DGtot ---
    xi_eq  = get_xieq_PT(P,T,n_0,nus,refdata)
    minDG  = get_G_PT(xi_eq,P,T,n_0,nus,refdata)
    minDG  = minDG/(R*T)

    # ---- minimum of DGmix ---
    min2xi  = xis[np.argmin(DDGmix)]
    min2DG  = np.min(DDGmix)/(R*T)

    # ---- Number of moles and pressure at equilibrium ----
    n_eq = n_0 + xi_eq * nus
    y_eq = n_eq / n_eq.sum()
    p_eq = P * y_eq
    V_eq = (n_eq.sum())*R*T/P

    # ---- Calculation of Kp and Ky ----
    # (a) using molar fractions and partial pressures (from xi_eq)
    Kp_v1, Ky_v1 = 1.0, 1.0
    for p_eq_j,nu_j in zip(p_eq,nus): Kp_v1 *= (p_eq_j/P_o)**nu_j
    for y_eq_j,nu_j in zip(y_eq,nus): Ky_v1 *= y_eq_j**nu_j
    # (b) using deltaG values
    Ky_v2 = np.exp(-Gast/(R*T))
    Kp_v2 = np.exp(-DGo /(R*T))
    # (c) directly from xi_eq (specific case A --> 2B)
    Ky_v3 = 4*xi_eq * xi_eq / (1-xi_eq*xi_eq)

    # --- Plot data ---
    fig = plt.figure(figsize=(5.50,3.67))
    plt.plot(xis, DGlin/(R*T), color='b',ls="--",label=r"$f = \xi \cdot \Delta_{r} G^{\ast}$")
    plt.plot(xis,DDGmix/(R*T), color='r',ls=":" ,label=r"$f = \Delta_{\rm mix}G(\xi) - \Delta_{\rm mix}G(0)$")
    plt.plot(xis, DGtot/(R*T), color='k',ls="-" ,label=r"$f = G(\xi) - G(0)$")
    plt.plot( xi_eq ,minDG ,'ko' )
    plt.plot( min2xi,min2DG,'rx' )

    # some formatting
    plt.xticks(fontsize=FONTSIZE[1])
    plt.yticks(fontsize=FONTSIZE[1])
    plt.xlabel(r'$\xi \;\; \mathrm{[mol]}$'           , fontsize=FONTSIZE[2])
    plt.ylabel(r'$f \; / \; (RT) \;\; \mathrm{[mol]}$', fontsize=FONTSIZE[2])
    plt.title(fr'T={T:.0f} K, p={P/1E5:.2f} bar')

    # secondary grid in x-axis
    ax = plt.gca()
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(0.1))
    ax.grid(True, which='minor', axis='x', alpha=0.15)
    ax.tick_params(axis='x', which='minor',bottom=False,top=False,length=0)
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.legend(loc="best", fontsize=FONTSIZE[0])

    # --- update global variable: last_fig ---
    global last_fig
    last_fig = plt.gcf()

    # --- Show and close figure ---
    plt.show()
    plt.close()

    # ---- Print info ----
    PVT_0  = (P,V_0 ,T)
    PVT_eq = (P,V_eq,T)
    # print(fr"Value for Delta_r{{G}}^o({T:.2f} K) = {DGo/1000:.2f} kJ/mol")
    # print(fr"Value for Delta_r{{G}}^*({T:.2f} K) = {Gast/1000:.2f} kJ/mol")
    print_info_eq("G",PVT_0,PVT_eq,molecules,xi_eq,n_eq,y_eq,p_eq,P,Kp_v1,Kp_v2,Ky_v1,Ky_v2)
# --------------------------------------------- #
def plot_DA_VT(T,V, fixed_args):

    molecules,nus,n_0,xis,refdata = fixed_args

    # ---- Terms in DAtot ----
    DAtot, term_lineal, term_nolin, P_xi = get_A_VT(xis,V,T,n_0,nus,refdata)
    # ---- minimum of DAtot ---
    idx_eq = np.argmin(DAtot)
    xi_eq  = xis[idx_eq]
    P_eq   = P_xi[idx_eq]
    minDA  = np.min(DAtot)/(R*T)

    # ---- minimum of DAtot ---
    xi_eq  = get_xieq_VT(V,T,n_0,nus,refdata)
    minDA  = get_A_VT(xi_eq,V,T,n_0,nus,refdata)[0]/(R*T)

    # ---- minimum of non-lineal term ---
    min2xi  = xis[np.argmin(term_nolin)]
    min2yy  = np.min(term_nolin)/(R*T)

    # ---- Number of moles and pressure at equilibrium ----
    n_eq = n_0 + xi_eq * nus
    y_eq = n_eq / n_eq.sum()
    p_eq = P_eq * y_eq

    # ---- Calculation of Kp and Ky ----
    # (a) using molar fractions and partial pressures (from xi_eq)
    Kp_v1, Ky_v1 = 1.0, 1.0
    for p_eq_j,nu_j in zip(p_eq,nus): Kp_v1 *= (p_eq_j/P_o)**nu_j
    for y_eq_j,nu_j in zip(y_eq,nus): Ky_v1 *= y_eq_j**nu_j
    # (b) using deltaG values
    DGo          = get_DGo(T,refdata)
    Kp_v2, Ky_v2 = np.exp(-DGo /(R*T)), None

    # --- Plot data ---
    plt.figure(figsize=(5.50,3.67))

    plt.plot(xis,term_lineal/(R*T), color='b',ls="--",label=r"$f = \xi \cdot \Delta_{r} G^{o}$")
    plt.plot(xis,term_nolin /(R*T), color='r',ls=":",label=r"$f = V \sum_i \left[ p_i \ln\frac{p_i}{e p^\circ} - p_i(0) \ln \frac{p_i(0)}{e p^\circ} \right]$")
    plt.plot(xis,      DAtot/(R*T), color='k',ls="-" ,label=r"$f = A(\xi) - A(0)$")

    plt.plot( xi_eq ,minDA ,'ko' )
    plt.plot(min2xi ,min2yy ,'rx' )

    plt.xticks(fontsize=FONTSIZE[1])
    plt.yticks(fontsize=FONTSIZE[1])
    plt.xlabel(r'$\xi \;\; \mathrm{[mol]}$'           , fontsize=FONTSIZE[2])
    plt.ylabel(r'$f \; / \; (RT) \;\; \mathrm{[mol]}$', fontsize=FONTSIZE[2])
    plt.title(fr'T={T:.0f} K,  V={V*1000:.2f} L')

    # secondary grid in x-axis
    ax = plt.gca()
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(0.1))
    ax.grid(True, which='minor', axis='x', alpha=0.15)
    ax.tick_params(axis='x', which='minor',bottom=False,top=False,length=0)
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.legend(loc="best",fontsize=FONTSIZE[0])

    # --- update global variable: last_fig ---
    global last_fig
    last_fig = plt.gcf()

    # --- Show and close figure ---
    plt.show()
    plt.close()

    # ---- Print info ----
    P_0 = n_0.sum() * R*T/V
    PVT_0  = (P_0 ,V,T)
    PVT_eq = (P_eq,V,T)
    print_info_eq("A",PVT_0,PVT_eq,molecules,xi_eq,n_eq,y_eq,p_eq,P_eq,Kp_v1,Kp_v2,Ky_v1,Ky_v2)
# --------------------------------------------- #
def plot_vib_average(Tmin,Tmax,freqmin,freqmax,freq=None):
    freqs    = np.linspace(freqmin,freqmax,51)
    averages = vib_contri_avera(freqs,Tmin,Tmax)
    yy_inf   = vib_contribution(freqs,Tmin)
    yy_sup   = vib_contribution(freqs,Tmax)
    plt.plot(freqs,yy_inf  ,'r--',label=rf"$T$={Tmin:.2f} K",zorder=2)
    plt.plot(freqs,yy_sup  ,'b--',label=rf"$T$={Tmax:.2f} K")
    plt.plot(freqs,averages,'k-' ,label=rf"average")
    plt.xlabel(r"Vib. frequency (cm$^{-1}$)"     ,fontsize=14)
    plt.ylabel(r"Vib. contribution, $n^{V^\ast}$",fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # --- update global variable: last_fig ---
    global last_fig
    last_fig = plt.gcf()

    if freq is not None:
       # limits so far
       xlim = plt.gca().get_xlim()
       ylim = plt.gca().get_ylim()
       # get average contribution for selected frequency
       aver =  vib_contri_avera(freq,Tmin,Tmax)
       # plot data for selected frequency
       plt.plot([freq,freq],[ylim[0],aver],'--',color="grey",zorder=1)
       plt.plot([xlim[0],freq],[aver,aver],'--',color="grey",zorder=1)
       plt.plot(freq,aver,'o',color="grey",label=rf"$n^{{V^\ast}} = {aver:.3f}$")
       # keep origonal limits
       plt.xlim(xlim)
       plt.ylim(ylim)
    plt.legend(loc="best",fontsize=11)

    # --- Show and close figure ---
    plt.show()
    plt.close()
# ============================================= #




In [ ]:
#@title <small> 💻 Load Auxiliary Functions Specific for the N2O4/NO2 equilibrium <small> { display-mode: "form" }

# ============================================= #
# ---- Specific for N2O4 <-> 2NO2 (PART 1) ---- #
# ============================================= #
def load_n2o4_2no2():
    # from Atkins' Physical Chemistry
    T_ref    = 298
    DHo_ref  =  57.20 * 1E3
    DSo_ref  = 175.83
    DCPo_ref =  -2.88
    #-------------------------------------------
    refdata = (DHo_ref,DSo_ref,DCPo_ref,T_ref)
    #-------------------------------------------
    molecules     = ["N2O4","NO2"]
    nus           = np.array([-1,2])
    #-------------------------------------------
    GEOMINFO      = {"N2O4":[(1,5),(4,5),(1,5,3)] , "NO2":[(0,1),(1,0,2)]}
    #-------------------------------------------
    n_0           = np.array([1.00, 0.00])
    #-------------------------------------------
    return refdata,molecules,nus,n_0,GEOMINFO
# --------------------------------------------- #
def get_xieq_PT_N2O4(T,P,nA0,nB0):
    refdata = load_n2o4_2no2()[0]
    dG0     = get_DGo(T,refdata)
    Kp      = np.exp(-dG0/R/T)
    alpha   = Kp * P_o/P
    # solution for second-order equation, knowing that (-nB0/2<= xi <= nA0)
    xi_eq   = (-nB0+(2*nA0+nB0)*np.sqrt(alpha/(alpha+4)))/2
    return xi_eq
# --------------------------------------------- #
def get_xieq_VT_N2O4(T,V,nA0,nB0):
    refdata = load_n2o4_2no2()[0]
    dG0     = get_DGo(T,refdata)
    Kp      = np.exp(-dG0/R/T)
    beta    = Kp * (P_o*V/R/T)
    # solution for second-order equation, knowing that (-nB0/2<= xi <= nA0)
    xi_eq   = (-4*nB0-beta+np.sqrt((8*(2*nA0+nB0)+beta)*beta))/8
    return xi_eq
# --------------------------------------------- #
def yB_to_xi_N2O4(yB):
    refdata,molecules,nus,n_0,GEOMINFO = load_n2o4_2no2()
    xi = (yB*n_0.sum() - n_0[1])/(2-yB)
    return xi
# --------------------------------------------- #
def intercept_getGm_N2O4(T,P,yB):
    refdata,molecules,nus,n_0 = load_n2o4_2no2()[0:4]
    xi     = yB_to_xi_N2O4(yB)
    ntot   = n_0.sum() + xi
    Gtot   = get_G_PT(xi,P,T,n_0,nus,refdata)
    Gm     = Gtot/ntot
    return Gm, Gtot, ntot
# --------------------------------------------- #
def intercept_getline_N2O4(T,P,yB_i):
    refdata,molecules,nus,n_0 = load_n2o4_2no2()[0:4]
    #partial pressures and quotient of reaction
    yA_i  = 1-yB_i
    pB_i  = P*yB_i
    pA_i  = P*yA_i
    Qp    = (pB_i/P_o)**2 / (pA_i/P_o)
    # value for xi_i
    xi_i  = yB_to_xi_N2O4(yB_i)
    # delta_r G^o (T)
    dGo_T = get_DGo(T,refdata)
    # G_tot
    Gm_i, Gtot_i, ntot_i = intercept_getGm_N2O4(T,P,yB_i)
    # get slope (m)
    dxidyB = (2*n_0.sum() - n_0[1]) / (2-yB_i)**2
    dGdxi  = dGo_T + R*T*np.log(Qp)
    m      = 1/ntot_i * dxidyB * (dGdxi - Gm_i)
    # Point of the line
    xx     = yB_i
    yy     = (Gm_i - intercept_getGm_N2O4(T,P,0)[0])/(R*T)
    # get intercept (b)
    m      = m / (R*T)
    b      = yy - m*xx
    return m,b, (xx,yy)
# --------------------------------------------- #
def plot_intercept_N2O4(T,P,yB):
    nA0,nB0 = load_n2o4_2no2()[3]
    # equilibrium
    xi_eq   = get_xieq_PT_N2O4(T,P,nA0,nB0)
    yB_eq   = (n_0[1]+2*xi_eq)/(n_0.sum() + xi_eq)
    Gtot_eq = intercept_getGm_N2O4(T,P,yB_eq)[1]
    # Get data in terms of yB_i
    lGm,lGtot,lntot = [],[],[]
    list_yB         = np.linspace(0.0,1.0,71)
    for yB_i in list_yB:
        Gm_i, Gtot_i, ntot_i = intercept_getGm_N2O4(T,P,yB_i)
        lntot.append(ntot_i)
        lGtot.append(Gtot_i)
    # control point
    if yB == 0.0: yB = ZERO4
    if yB == 1.0: yB = 1 - ZERO4
    # equation for tangent of Gm
    m,b,(xx1,yy1) = intercept_getline_N2O4(T,P,yB)
    # values at the selected point
    Gm, Gtot, ntot = intercept_getGm_N2O4(T,P,yB)
    # apply reference
    Gtot    =   (Gtot    - lGtot[0])/(R*T)
    Gtot_eq =   (Gtot_eq - lGtot[0])/(R*T)
    lGtot   = [(lGtot[i] - lGtot[0])/(R*T) for i in range(len(lGtot))]
    lGm     = [lGtot[i]/lntot[i]           for i in range(len(lGtot))]
    # --- Create side-by-side axes ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    # -- left plot --
    ax1.plot(list_yB,lGtot,'k-')
    ax1.plot(yB_eq,Gtot_eq,'ko',zorder=1)
    ax1.plot(yB   ,Gtot   ,'ro',zorder=2)
    for ii in ["x","y"]: ax1.tick_params(axis=ii,labelsize=FONTSIZE[1])
    ax1.set_xlim( 0.0 , 1.0 )
    ax1.set_xlabel(r"$y_{\rm NO_2}$",fontsize=FONTSIZE[2])
    ax1.set_ylabel(r"$[G(y_{\rm NO_2})-G(0)]\cdot (RT)^{-1} \;\; \mathrm{[mol]}$",fontsize=FONTSIZE[2])
    # -- right plot --
    xx2 = [ZERO4,1-ZERO4]
    yy2 = [m*xx_i + b for xx_i in xx2]
    ylim1 = min(lGm)
    ylim2 = max(lGm)
    delta = ylim2-ylim1
    ylim1 = ylim1 - delta*0.1
    ylim2 = ylim2 + delta*0.1
    ax2.plot(list_yB,lGm,'k-')
    ax2.plot(xx1,yy1,'ro')
    ax2.plot(xx2,yy2,'r--',label=rf"${m:+.5f} \cdot y_{{\rm NO_2}} {b:+.5f}$")
    for ii in ["x","y"]: ax2.tick_params(axis=ii,labelsize=FONTSIZE[1])
    ax2.set_xlim( 0 , 1 )
    ax2.set_ylim(ylim1,ylim2)
    ax2.set_xlabel(r"$y_{\rm NO_2}$",fontsize=FONTSIZE[2])
    ax2.set_ylabel(r"$[G_{\rm m}(y_{\rm NO_2})- G_{\rm m}(0)] \cdot (RT)^{-1}$",fontsize=FONTSIZE[2])
    ax2.legend(loc="upper center",fontsize=FONTSIZE[0])
    # --- update global variable: last_fig ---
    global last_fig
    last_fig = plt.gcf()
    # --- Show and close figure ---
    plt.show()
    plt.close()
    print(rf"Equilibrium at y(NO2) = {yB_eq:.7f}")
# --------------------------------------------- #
def plot_3DeqPT_N2O4(T,P):

    refdata,molecules,nus,n_0 = load_n2o4_2no2()[0:4]

    # calculate data for each plot
    Z1 = get_xieq_PT_N2O4(T,P,n_0[0],n_0[1])
    Z2 = get_G_PT(Z1,P,T,n_0,nus,refdata)/(R*T)

    # Create figure with two subplots
    # title1 = r'$(a) \;\; z: \;\; \xi_{\mathrm{eq}} \;\; \text{(mol)}$'
    # title2 = r'$(b) \;\; z: \;\; [G(\xi_{\mathrm{eq}}) - G(0)] / RT  \;\; \text{(mol)}$'
    title1 = "(a)  z:  <i>ξ</i><sub>eq</sub> (mol)"
    title2 = "(b)  z:  [<i>G</i>(<i>ξ</i><sub>eq</sub>) − <i>G</i>(0)] / <i>RT</i>  (mol)"
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'surface'}, {'type': 'surface'}]],
        subplot_titles=(title1,title2))

    fig.add_trace(go.Surface(x=P/1E5, y=T, z=Z1, colorscale='Viridis', showscale=False),row=1, col=1)
    fig.add_trace(go.Surface(x=P/1E5, y=T, z=Z2, colorscale='Plasma' , showscale=False),row=1, col=2)

    # formating
    fig.update_layout(
        width=1000, height=450,
        margin=dict(l=0, r=0, t=0, b=0),
        scene=dict(
          xaxis=dict(title=dict(text='p (bar)',font=dict(size=18))),
          yaxis=dict(title=dict(text='T (K)'  ,font=dict(size=18))),
          zaxis=dict(title=dict(text=''       ,font=dict(size=18))),
          domain=dict(x=[0.00, 0.50], y=[0, 1]),
          camera=dict(eye=dict(x=1.6, y=1.6, z=0.9))),
        scene2=dict(
          xaxis=dict(title=dict(text='p (bar)',font=dict(size=18))),
          yaxis=dict(title=dict(text='T (K)'  ,font=dict(size=18))),
          zaxis=dict(title=dict(text=''       ,font=dict(size=18))),
          domain=dict(x=[0.50, 1.00], y=[0, 1]),
          camera=dict(eye=dict(x=1.6, y=1.6, z=0.9)))
    )

    # Ajust position of plot titles
    for ann in fig['layout']['annotations']:
        ann['y'] -= 0.15
        ann['font'] = dict(size=16)

    fig.show(config={"toImageButtonOptions": {"format": "svg","filename": "equilibrium_surfaces","scale": 2}})
# ============================================= #

# ============================================= #
# ---- Specific for N2O4 <-> 2NO2 (PART 3) ---- #
# ============================================= #
def get_constants_N2O4(T):
    # Gibbs free energy from experimental data
    DGo   = get_DGo(T,load_n2o4_2no2()[0])
    # Equilibrium constant(s)
    Kp_o  = np.exp(-DGo/R/T)
    Kc_o  = Kp_o * P_o/(c_o*R*T) # adimensional
    Kc    = Kc_o * c_o           # mol/m^3
    # Forward rate constant
    kfw   = arrhenius_A * np.exp(-arrhenius_B/T)
    # Backward rate constant
    kbw  = kfw/Kc
    # String with data
    string  = rf"   * Constants for the reaction at {T:.2f} K" + "\n"
    string += "\n"
    string += rf"     equilibrium constant   Kp^o = {Kp_o:.3E}" + "\n"
    string += rf"     equilibrium constant   Kc^o = {Kc_o:.3E}" + "\n"
    string += rf"     forward  rate constant kfw  = {kfw:.3E} 1/s" + "\n"
    string += rf"     backward rate constant kbw  = {kbw:.3E} m^3 / mol / s" + "\n"
    string += "\n"
    # return data
    return DGo,Kp_o,Kc_o,kfw,kbw,string
# --------------------------------------------- #
def xi_to_data_N2O4(xi,T0,P0,V0,yA0,scenario):
    # values at xi=0
    n0 = P0*V0/(R*T0)
    nA0 = yA0*n0
    nB0 = n0-nA0
    # values at xi
    nA = nA0 -   xi
    nB = nB0 + 2*xi
    n  = n0  +   xi
    yA = nA/n
    yB = nB/n
    if   "VT" in scenario: P,V = n*R*T0/V0, V0
    elif "PT" in scenario: P,V = P0 , n*R*T0/P0
    else: raise Exception
    cA = nA/V
    cB = nB/V
    pA = yA*P
    pB = yB*P
    # quotient of reaction
    Qp = np.where(pA == 0,np.inf,(pB/P_o)**2 / (pA/P_o))
    # energy of interest
    if "VT" in scenario: energy = get_A_VT_N2O4(xi,V0,T0,nA0,nB0)
    if "PT" in scenario: energy = get_G_PT_N2O4(xi,P0,T0,nA0,nB0)
    # refdata = load_n2o4_2no2()[0]
    # if "PT" in scenario: energy = get_G_PT(xi,P0,T0,np.array([nA0,nB0]),np.array([-1,2]),refdata)
    # if "VT" in scenario: energy = get_A_VT(xi,V0,T0,np.array([nA0,nB0]),np.array([-1,2]),refdata)
    # return data
    return (nA,nB),(pA,pB),(yA,yB),(cA,cB),(n,P,V),Qp,energy
# --------------------------------------------- #
def xi2time_PT_N2O4(xi,xi1,xi2,kfw,alpha):
    beta  = kfw + 4*alpha
    s     = np.sqrt(kfw/beta)
    term1 = (xi1-xi)/xi1
    term2 = (xi2-xi)/xi2
    texp  = term1**(1+s) / term2**(1-s)
    t     = -np.log(texp)/(2*s*beta)
    return t
# --------------------------------------------- #
def xi2time_VT_N2O4(xi,xi1,xi2,kbw,V0):
    t  = np.log( abs(xi1/xi2 * (xi-xi2)/(xi-xi1))  )
    t *= V0/(4*kbw*(xi1-xi2))
    return t
# --------------------------------------------- #
def get_G_PT_N2O4(xi,P,T,nA0,nB0):
    #Delta_r{G}^o(T)
    DGo_T   = get_constants_N2O4(T)[0]
    #Delta_r{G}^*(T)
    DGast   = DGo_T + R*T*np.log(P/P_o)
    # mix term
    n0      = nA0 + nB0
    nA      = nA0 -   xi
    nB      = nB0 + 2*xi
    n       = nA  + nB
    DDGmix  = 0.0
    if nA  != 0.0: DDGmix += nA *np.log(nA /n )
    if nB  != 0.0: DDGmix += nB *np.log(nB /n )
    if nA0 != 0.0: DDGmix -= nA0*np.log(nA0/n0)
    if nB0 != 0.0: DDGmix -= nB0*np.log(nB0/n0)
    DDGmix *= R*T
    # return G(xi) - G(0)
    DGtot   = DGast * xi + DDGmix
    return DGtot
# --------------------------------------------- #
def get_A_VT_N2O4(xi,V,T,nA0,nB0):
    # initial conditions
    n0      = nA0 + nB0
    p0      = n0*R*T/V
    # current conditions
    nA      = nA0 -   xi
    nB      = nB0 + 2*xi
    n       = nA  + nB
    p       = n *R*T/V
    #Delta_r{G}^o(T)
    DGo_T   = get_constants_N2O4(T)[0]
    # pressure term
    termP   = p *np.log(p/P_o/np.e)
    termP  -= p0*np.log(p0/P_o/np.e)
    # mix term
    DDGmix  = 0.0
    if nA  != 0.0: DDGmix += nA *np.log(nA /n )
    if nB  != 0.0: DDGmix += nB *np.log(nB /n )
    if nA0 != 0.0: DDGmix -= nA0*np.log(nA0/n0)
    if nB0 != 0.0: DDGmix -= nB0*np.log(nB0/n0)
    DDGmix *= R*T
    # return A(xi) - A(0)
    DAtot   = DGo_T * xi + V*termP + DDGmix
    return DAtot
# --------------------------------------------- #
def datatoinfo_N2O4(T0,p0,V0,yA0,xi,scenario):
    # constants
    Kp_o = get_constants_N2O4(T0)[1]
    # calculate more magnitudes
    (nA,nB),(pA,pB),(yA,yB),(cA,cB),(n,P,V),Qp,E = xi_to_data_N2O4(xi,T0,p0,V0,yA0,scenario)
    # string with information
    string  = rf"       (P , V , T) = ({P*1E-5:6.2f} bar , {V*1E3:6.2f} L , {T0:6.2f} K)"+"\n"
    string += "\n"
    string += rf"       num. moles  = {n:6.3f} mol"+"\n"
    string += rf"       extent (xi) = {xi:6.3f} mol"+"\n"
    string += "\n"
    if "VT" in scenario: string += rf"       A(xi)-A(0)  = {E/(R*T0):8.2E}*(RT) mol"+"\n"
    if "PT" in scenario: string += rf"       G(xi)-G(0)  = {E/(R*T0):8.2E}*(RT) mol"+"\n"
    string += "\n"
    string += rf"       data for N2O4"+"\n"
    string += rf"         - number of moles  = {nA:6.3f} mol"+"\n"
    string += rf"         - mole fraction    = {yA:6.3f} mol"+"\n"
    string += rf"         - partial pressure = {pA*1E-5:6.3f} bar"+"\n"
    string += rf"         - concentration    = {cA/1000:8.2E} M"+"\n"
    string += "\n"
    string += rf"       data for NO2"+"\n"
    string += rf"         - number of moles  = {nB:6.3f} mol"+"\n"
    string += rf"         - mole fraction    = {yB:6.3f} mol"+"\n"
    string += rf"         - partial pressure = {pB*1E-5:6.3f} bar"+"\n"
    string += rf"         - concentration    = {cB/1000:8.2E} M"+"\n"
    string += "\n"
    if pA == 0.0: string += rf"       ==> Qp^o = infinity"+"\n"
    else        : string += rf"       ==> Qp^o = {(pB*pB)/(pA*P_o):.3E}"+"\n"
    string += rf"       ==> Kp^o = {Kp_o:.3E}"+"\n"
    string += "\n"
    return string
# --------------------------------------------- #
def kinetics_N2O4(T0,P0,V0,yA0,scenario):
    # --- global variable to update ---
    global last_info
    # which scenario
    if   "PT" in scenario: scenario = "PT"
    elif "VT" in scenario: scenario = "VT"
    else: raise Exception
    # Get data for this reaction
    DGo,Kp_o,Kc_o,kfw,kbw,STRING = get_constants_N2O4(T0)
    # Determine xieq, xi1 and xi2
    (nA0,nB0) = xi_to_data_N2O4(0,T0,P0,V0,yA0,scenario)[0]
    if scenario == "VT":
       xieq = get_xieq_VT_N2O4(T0,V0,nA0,nB0)
       Kc   = kfw/kbw
       lamb = 4*(kbw/V0)
       a0   = (nB0**2/4 - nA0*V0*Kc/4)
       a1   = (nB0      +     V0*Kc/4)
       xi1  = (-a1 + np.sqrt(a1**2-4*a0))/2
       xi2  = (-a1 - np.sqrt(a1**2-4*a0))/2
    if scenario == "PT":
       xieq  = get_xieq_PT_N2O4(T0,P0,nA0,nB0)
       alpha = kbw*P0/(R*T0)
       beta  = kfw + 4*alpha
       s     = np.sqrt(kfw/beta)
       xi1   = (-nB0+(2*nA0+nB0)*s)/2
       xi2   = (-nB0-(2*nA0+nB0)*s)/2
    # Calculate from xi = 0 to xieq
    xis    = np.linspace(0,REL_XI_EQ*xieq,NPOINTSXI)
    if scenario == "VT": times = [xi2time_VT_N2O4(xi,xi1,xi2,kbw,V0   ) for xi in xis]
    if scenario == "PT": times = [xi2time_PT_N2O4(xi,xi1,xi2,kfw,alpha) for xi in xis]
    # for t_i,xi_i in zip(times,xis): print(t_i,xi_i)
    # String with information
    STRING += rf"   * Initial conditions:"+"\n\n"
    STRING += datatoinfo_N2O4(T0,P0,V0,yA0,0   ,scenario)
    STRING += rf"   * At equilibrium:"+"\n\n"
    STRING += datatoinfo_N2O4(T0,P0,V0,yA0,xieq,scenario)
    last_info = STRING
    # plot data
    plot_kinetics_N2O4(np.array(times),xis,xieq,T0,P0,V0,yA0,scenario)
# ============================================= #
def plot_kinetics_N2O4(times,xis,xieq,T0,P0,V0,yA0,scenario):

    # select good units for time (among secs, milisecs, microsecs and nanosecs)
    for unitst,factor in [("s",1E0) , ("ms",1E3) , ("$\\mu$s",1E6) , ("ns",1E9)]:
        last_t = times[-1]*factor
        if last_t > 0.5: break

    dataeq = xi_to_data_N2O4(xieq,T0,P0,V0,yA0,scenario)
    yA,yB = [],[]
    Qp    = []
    AG    = []
    V     = []
    P     = []
    nA    = []
    for xi_i in xis:
        tni,tpi,tyi,tci,(n_i,P_i,V_i),Qp_i,E_i = xi_to_data_N2O4(xi_i,T0,P0,V0,yA0,scenario)
        nA.append(tni[0])
        yA.append(tyi[0])
        yB.append(tyi[1])
        Qp.append(Qp_i)
        AG.append(E_i)
        V.append(V_i)
        P.append(P_i)

    plt.rcParams['text.usetex'] = True
    fig, axs = plt.subplots(2, 3, figsize=(12,6))
    fig.suptitle(rf'$(P,V,T)_0 = ({P0*1E-5:.2f} \; {{\rm bar}},{V0*1E3:.2f} \; {{\rm L}},{T0:.2f} \; {{\rm K}})$; $y_{{\rm N_2O_4}}(0)={yA0:.2f}$', fontsize=FONTSIZE[2])
    # -------------------------------------
    # (a) Population
    # -------------------------------------
    axs[0, 0].plot(times*factor,yA,color='k',label=r'i=N$_2$O$_4$')
    axs[0, 0].axhline(y=dataeq[2][0],color="k",ls=":",zorder=1)

    axs[0, 0].plot(times*factor,yB,color='r',label=r'i=NO$_2$')
    axs[0, 0].axhline(y=dataeq[2][1],color="r",ls=":",zorder=1)

    axs[0, 0].yaxis.set_major_locator(mticker.MultipleLocator(0.2))
    axs[0, 0].set_ylabel(r'$y_i$',fontsize=FONTSIZE[2])
    axs[0, 0].set_xlabel(rf'Time ({unitst:s})',fontsize=FONTSIZE[2])
    axs[0, 0].set_title('(a)')
    axs[0, 0].legend(frameon=False)

    # -------------------------------------
    # (b) ξ vs time
    # -------------------------------------
    nA0,nB0 = xi_to_data_N2O4(0,T0,P0,V0,yA0,scenario)[0]
    xlim1   = -nB0/2
    xlim2   = +nA0
    axs[0, 1].plot(times*factor,xis,ls='-',color='k')
    axs[0, 1].axhline(y=xieq,ls=":",color="k",zorder=1)

    axs[0, 1].set_ylabel('$\\xi$ (mol)',fontsize=FONTSIZE[2])
    # axs[0, 1].set_ylim(xlim1,xlim2)
    axs[0, 1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

    axs[0, 1].set_xlabel(rf'Time ({unitst:s})',fontsize=FONTSIZE[2])
    axs[0, 1].set_title('(b)')

    # -------------------------------------
    # (c) P(or V) vs time
    # -------------------------------------
    if "PT" in scenario:
        axs[0, 2].plot(times*factor,np.array(V)*1E3 ,ls='-',color='k')
        axs[0, 2].set_ylabel('$V$ (L)',fontsize=14)
    if "VT" in scenario:
        axs[0, 2].plot(times*factor,np.array(P)*1E-5,ls='-',color='k')
        axs[0, 2].set_ylabel('$P$ (bar)',fontsize=FONTSIZE[2])
    axs[0, 2].set_xlabel(rf'Time ({unitst:s})',fontsize=FONTSIZE[2])
    axs[0, 2].set_title('(c)')

    # -------------------------------------
    # (d) Q vs time
    # -------------------------------------
    xx,yy=factor*times[1:], Qp[1:]
    axs[1, 0].plot(xx,yy,ls='-',color='k',zorder=2)
    axs[1, 0].axhline(y=dataeq[5],ls=":" ,color="k",zorder=1)
    axs[1, 0].set_ylabel('$Q_p^\\circ$',fontsize=FONTSIZE[2])
    axs[1, 0].set_xlabel(rf'Time ({unitst:s})',fontsize=FONTSIZE[2])
    axs[1, 0].set_title('(d)')

    # -------------------------------------
    # (e) dξ/dt vs time
    # -------------------------------------
    Kpo,Kco,kfw = get_constants_N2O4(T0)[1:4]
    dxidt_ana       = kfw*np.array(nA)*(1-Qp/Kpo)
    axs[1, 1].plot(times*factor,dxidt_ana/factor,ls='-',color='k')
    # dxidt_num       = np.gradient(xis,times)
    # axs[1, 1].plot(times*factor,dxidt_num/factor,'rx')
    axs[1, 1].set_ylabel(rf'd$\xi$/d$t$ (mol/{unitst:s})',fontsize=FONTSIZE[2])
    axs[1, 1].set_xlabel(rf'Time ({unitst:s})',fontsize=FONTSIZE[2])
    axs[1, 1].set_title('(e)')

    # -------------------------------------
    # (f) G/RT vs time
    # -------------------------------------
    yy = [Ei/(R*T0) for Ei in AG]
    axs[1, 2].plot(times*factor,yy,color='k')
    axs[1, 2].axhline(y=dataeq[6]/(R*T0),ls=":",color="k",zorder=1)
    if "PT" in scenario: key = "G"
    if "VT" in scenario: key = "A"
    axs[1, 2].set_ylabel(rf'$\left({key:s}(\xi)-{key:s}(0)\right) / (RT)$ (mol)',fontsize=FONTSIZE[2])
    axs[1, 2].set_xlabel(rf'Time ({unitst:s})',fontsize=FONTSIZE[2])
    axs[1, 2].set_title('(f)')

    # --- update global variable: last_fig ---
    plt.tight_layout()
    global last_fig
    last_fig = fig
    # --- Show and close figure ---
    fig.subplots_adjust(wspace=0.3)
    # fig.set_size_inches(9.0,6.6)
    plt.show()
    plt.close(fig)
# ============================================= #




### **Loading the reaction**  


Execute the following cell to load the reaction.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
MOLECULES,NUS,n_0 = load_n2o4_2no2()[1:4]
#------------------------------------------------------
string = reaction_to_string(NUS,MOLECULES)
#------------------------------------------------------
print(rf"The reaction to study is: {string:s}")
print("")
print(rf"Initial number of moles considered for each species:")
for molecule,n0_i in zip(MOLECULES,n_0):
    print(rf"   * {molecule:4s}: {n0_i:.3f} mol")
print("")
#------------------------------------------------------

### **1. A Chemical Thermodynamics Perspective**  


####
At 298 K, the following thermodynamic data are available for the reaction (see Atkins' *Physical Chemistry*, **2014**, Oxford University Press):
-  for $\rm N_2O_4$ $\Rightarrow$ $\Delta_f{H}^\circ = \;\; 9.16$ kJ mol$^{-1}$, $S^\circ = 304.29$ J mol$^{-1}$ K$^{-1}$ and $C_{p,m}^\circ = 77.28$ J mol$^{-1}$ K$^{-1}$;
-  for $\rm NO_2$ $\;$ $\Rightarrow$ $\Delta_f{H}^\circ = 33.18$ kJ mol$^{-1}$, $S^\circ = 240.06$ J mol$^{-1}$ K$^{-1}$ and $C_{p,m}^\circ = 37.20$ J mol$^{-1}$ K$^{-1}$.

From these values, we obtain:
- $\Delta_{r} H^{\circ}(T_{\rm ref}) = \;\;57.20$ kJ mol$^{-1}$,
- $\Delta_{r} S^{\circ}(T_{\rm ref}) \;= 175.83$ J mol$^{-1}$ K$^{-1}$ and
- $\Delta_{r} C_p^\circ(T_{\rm ref}) \;= -2.88$ J mol$^{-1}$ K$^{-1}$.

Load this data by executing the cell below.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
REFDATA = load_n2o4_2no2()[0]
#------------------------------------------------------
T_ref  = REFDATA[3]
#------------------------------------------------------
print(rf"Magnitudes of interest at Tref = {REFDATA[3]:.2f} K:")
print("")
print(rf"    Delta_r{{H}}^o  = {REFDATA[0]/1000:+8.2f} kJ/mol")
print(rf"    Delta_r{{S}}^o  = {REFDATA[1]:+8.2f}  J/(mol K)")
print(rf"    Delta_r{{Cp}}^o = {REFDATA[2]:+8.2f}  J/(mol K)")
print("")
#------------------------------------------------------

####
**(a) EFFECT OF THE TEMPERATURE ON $\Delta_r G^\circ$**

#####
For an ideal gas-phase reaction, the temperature dependence of the standard Gibbs free energy change, $\Delta_{r} G^\circ$, can be expressed as:

$$
\Delta_{r} G^{\circ}(T)= \Delta_{r} H^{\circ}(T_{\rm ref})-T \cdot \Delta_{r} S^{\circ}(T_{\rm ref})+\Delta_{r} C_p^\circ \cdot \left[ T-T_{\rm ref}+T \cdot \ln \left(\frac{T_{\rm ref}}{T}\right)\right]
 \tag{1}
$$

with $T_{\rm ref}$ being the reference $T$ and under the assumption that $\Delta_{r} C_p^\circ$ is independent of $T$. Notice that the corresponding equilibrium constant is then obtained as:

$$
K_p^\circ(T) = e^{-\Delta_{r} G^{\circ}(T)/(R \cdot T)}
 \tag{2}
$$

Examine the temperature dependence of $\Delta G^\circ$ (and of $K_p$) by executing the next cell. In the generated plots, the red dot indicates the value at $T_{\rm ref}$.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
T     = np.linspace(200,400,num=31)
DGo_T = plot_DGo_T(T,T_ref,REFDATA)
#------------------------------------------------------

#####
According to the Gibbs–Helmholtz equation, the following equality should hold:

$$
\frac{\Delta_{r} H^{\circ}}{T^2} = \frac{d}{dT}\left( -\frac{\Delta_{r} G^{\circ}}{T} \right)
\tag{3}
$$

This relationship can be tested by computing the numerical derivative of the term on the right-hand side and comparing it with the analytical expression on the left-hand side.

- The analytical expression for the enthalpy term as a function of temperature is:
$$
\Delta_{r} H^{\circ} = \Delta_{r} H^{\circ}(T_{\rm ref}) + \Delta_{r} C_{p}^\circ \cdot (T- T_{\rm ref})
\tag{4}
$$
Using this expression, we can easily plot $\Delta_{r} H^{\circ}/ T^2$.

- On the other side, the numerical derivative of $-\Delta_{r} G^{\circ}/T$ with respect to temperature can be directly computed from the data obtained in the execution of the previous cell.

Execute the next cell to check the Gibbs–Helmholtz relationship.

In [ ]:
#@title <small><small> { display-mode: "form" }

plot_gibbshelmholtz(T,DGo_T,REFDATA)

####
**(b) REACTION CONDUCTED AT CONSTANT _P_ AND _T_**

#####
$G$ changes dynamically as the reaction progresses. By knowing the initial conditions for the reaction, it is possible to track how $G$ varies with the extent of reaction ($\xi$), which is of special interest when the reaction takes place at constant $T$ and $p$.

Let us assume that the reaction vessel initially contains 1 mol of N$_2$O$_4$ (and none of NO$_2$). Execute the cell below, set the temperature and the total pressure, and analyze how $G$ changes with $\xi$:

In [ ]:
#@title <small><small> { display-mode: "form" }

#----  Get limit values for xi  ----
xi_min,xi_max = limits_xi(n_0,NUS)
#----     Get values for xi     ----
xis           = np.linspace(xi_min,xi_max,num=1001)
#----   Save info in a tuple    ----
fixed_args    = (MOLECULES,NUS,n_0,xis,REFDATA)
#-----------------------------------

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args     = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
T_slider = w.FloatSlider(value=298.00,min=200.00,max=400.00,step=1.00,description=r'T [K]'  , **args)
P_slider = w.FloatSlider(value=  1.00,min=  0.01,max=  3.00,step=0.01,description=r'p [bar]', **args)
ui       = w.VBox([T_slider,P_slider])

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: _on_download_clicked(b,"PT"))

# -------- slider ---> function --------
# notice that P is converted from bar to Pa with *1E5
out = w.interactive_output(lambda T, P, fixed_args: plot_DG_PT(T, P*1E5, fixed_args), {'T': T_slider, 'P': P_slider, 'fixed_args': w.fixed(fixed_args)})
display(w.VBox([ui, btn]), out)

#####
As you have observed, the equilibrium extent (and its corresponding Gibbs free energy) depends on the experimental temperature and pressure. This dependence can be visualized using a 3D plot, which enables us to examine (a) the equilibrium position, $\xi_{\rm eq}$, and (b) the change in Gibbs free energy from the initial state ($\xi = 0$) to the equilibrium state ($\xi_{\rm eq}$).

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
Ts  = np.linspace(200.15,400.15 ,201)
Ps  = np.linspace(0.01E5, 3.01E5,201)
P,T = np.meshgrid(Ps, Ts)
#------------------------------------------------------
plot_3DeqPT_N2O4(T,P)
#------------------------------------------------------

#####

Beyond the previous global view, it is also useful to analyze the equilibrium condition directly in terms of mixture composition through the molar Gibbs free energy, defined as

$$
G_m(\xi) =  G(\xi) / n(\xi) \tag{5}
$$

where both the total Gibbs free energy, $G$, and the total number of moles, $n$, depend on the extent of reaction.

For the binary mixture considered here, this can also be written as:

$$
G_m(\xi) =  y_{N_2O_4} \cdot \mu_{N_2O_4}  + y_{NO_2} \cdot \mu_{NO_2} \tag{6}
$$

By plotting $G_m$ as a function of $y_{NO_2}$, the equilibrium composition can also be identified. Notably, the equilibrium point does not coincide with the minimum of this curve. Execute the following cell and use the $y_{NO_2}$ slider to explore how the equilibrium point can be inferred from the $G_m$ plot (hint: it is related to the _intercept method_).

In [ ]:
#@title <small><small> { display-mode: "form" }

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args     = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True)
T_slider = w.FloatSlider(value=298.00,min=200.00,max=400.00,step=1.0000  ,description=r'T [K]'  ,readout_format='.2f',**args)
P_slider = w.FloatSlider(value=  1.00,min=  0.01,max=  3.00,step=0.0100  ,description=r'p [bar]',readout_format='.2f',**args)
yB_slider= w.FloatSlider(value=  0.50,min=  0.00,max=  1.00,step=0.000001,description=r'y(NO2)' ,readout_format='.6f',**args)

# --- Separator text between slider groups ---
separator = w.HTML(
    value=(
        "<div style='width:600px; margin:10px 0 6px 0;'>"
        "  <hr style='border:0; border-top:1px solid rgba(255,255,255,0.25); margin:0 0 8px 0;'>"
        "  <div style='text-align:center; font-size:13px; line-height:1.25; opacity:0.85;'>"
        "    Move <b>y(NO2)</b> to select a point on <b>G<sub>m</sub></b>.<br>"
        "    The <b>tangent</b> to <b>G<sub>m</sub></b> will be shown."
        "  </div>"
        "</div>"
    )
)

# sliders
ui = w.VBox([T_slider, P_slider,separator,yB_slider],
            layout=w.Layout(width='600px',display='flex',flex_flow='column',align_items='flex-start',justify_content='flex-start'))

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: _on_download_clicked(b,"intercept"))

# -------- slider ---> function --------
# notice that P is converted from bar to Pa with *1E5
out = w.interactive_output(lambda T, P, yB: plot_intercept_N2O4(T,P*1E5,yB), {'T': T_slider, 'P': P_slider, 'yB': yB_slider})
display(w.VBox([ui, btn]), out)

####
**(c) REACTION CONDUCTED AT CONSTANT _V_ AND _T_**

#####
Another case of interest is a reaction carried out in a reactor at constant $T$ and $V$. Under these conditions, the relevant thermodynamic potential is the Helmholtz free energy (A):

$$
A = G - pV = G - nRT
\tag{7}
$$

As in the previous case, let us assume that the reaction vessel initially contains 1 mol of N$_2$O$_4$ (and none of NO$_2$). Execute the cell below, set the temperature and the total volume, and analyze how $A$ changes with $\xi$:


In [ ]:
#@title <small><small> { display-mode: "form" }

#----  Get limit values for xi  ----
xi_min,xi_max = limits_xi(n_0,NUS)
#----     Get values for xi     ----
xis           = np.linspace(xi_min,xi_max,num=1001)
#----   Save info in a tuple    ----
fixed_args    = (MOLECULES,NUS,n_0,xis,REFDATA)
#-----------------------------------

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args     = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
T_slider = w.FloatSlider(value=298.00,min=200.00,max=400.00,step=1.00,description=r'T [K]', **args)
V_slider = w.FloatSlider(value= 24.78,min=  2.00,max=250.00,step=0.01,description=r'V [L]', **args)
ui       = w.VBox([T_slider,V_slider])

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: _on_download_clicked(b,"VT"))

# -------- slider ---> function --------
# V is converted from L to m3 with *1E-3
out = w.interactive_output(lambda T, V, fixed_args: plot_DA_VT(T, V*1E-3, fixed_args), {'T': T_slider, 'V': V_slider, 'fixed_args': w.fixed(fixed_args)})
display(w.VBox([ui, btn]), out)

### **2. A Statistical-Mechanical Perspective**  


#####
In the previous section, we analyzed the equilibrium using tabulated values of $\Delta_{r} H^{\circ}(T_{\rm ref})$, $\Delta_{r} S^{\circ}(T_{\rm ref})$ and $\Delta_{r} C_p^\circ(T_{\rm ref})$, which were then inserted into a macroscopic expression for $\Delta_{r} G^{\circ}(T)$. In this section, we will obtain these thermodynamic quantities from a molecular-level description using statistical mechanics.

For an ideal-gas mixture, the standard thermodynamic functions of each species can be derived from its molecular partition function, $q^\circ(T)$, which factorizes into translational, rotational, vibrational, and electronic contributions. From $q^\circ(T)$ we can compute $G^{\circ}_{m}(T)$, $H^{\circ}_{m}(T)$, $S^{\circ}_{m}(T)$, and $C^{\circ}_{p,m}(T)$ for each molecule. The reaction quantities then follow by applying the stoichiometric sum over reactants and products. In this way, the temperature dependence of
$\Delta_{r} G^{\circ}(T)$ is linked directly to molecular structure and vibrational spectra.

To start, we need initial molecular geometries for the reactant (N$_2$O$_4$) and product (NO$_2$). These structures will be used as input for subsequent electronic-structure calculations (geometry optimization and harmonic vibrational analysis), from which we will extract vibrational frequencies and other molecular parameters required to build the partition functions.

#### **(a) Guess geometries of reactants and products**

We will retrieve the geometry for N$_2$O$_4$ from PubChem[[🌐]](https://pubchem.ncbi.nlm.nih.gov/) by using its PubChem CID identifier[[🌐]](https://pubchem.ncbi.nlm.nih.gov/compound/25352):

* PubChem CID for N$_2$O$_4$ is "25352"

Run the cell below and enter this identifier. Do _not_ include quotation marks when entering the CID.
Once the geometry is retrieved, it will be displayed in an interactive viewer that allows you to rotate and inspect the molecule freely.

In [ ]:
#@title <small><small> { display-mode: "form" }

molecule = "N2O4"
GEOMINFO = load_n2o4_2no2()[4]
# ------------------------------------------------
if "MOLECULES_ID" not in globals(): MOLECULES_ID  = {k:None for k in MOLECULES}
# ------------------------------------------------
print("Fetching structures from PubChem 🔎 .\n")
print("Please enter the CID to search for:")
MOLECULES_ID[molecule] = input(rf"     * reactant ({molecule:s}): ").strip()
print("")

mol_id    = MOLECULES_ID[molecule]
xyz_guess = files_of_interest(molecule)[0]

# Get geometry
print(rf"Retrieving geometry for: {str(mol_id):s}")
print("")
symbols,coords,smiles = pubchem_cid(mol_id)

# Generate file
if symbols is None:
   print(rf"     - ERROR: unable to get geometry for: '{str(mol_id):s}'")
   print("")
else:
   data_2_xyz(symbols,coords,xyz_guess,smiles)
   print(rf"     - geometry stored in '{xyz_guess:s}'")
   info = geometric_info_xyz(xyz_guess,GEOMINFO[molecule])
   print(info)
   print("")
   view = create_visualization_xyz(xyz_guess)
   view.show()

#####

For NO$_2$ we will use an alternative way to obtain a reference structure: building the molecule from its SMILES representation[[🌐]](https://en.wikipedia.org/wiki/Simplified_Molecular_Input_Line_Entry_System) using with the **Rdkit** toolkit[[🌐]](https://www.rdkit.org/). You can find the SMILES for nitrogen dioxide online (including on Wikipedia[[🌐]](https://en.wikipedia.org/wiki/Nitrogen_dioxide)):

*  SMILES for NO$_2$: "[N+]\(=O)[O-]":

Run the following cell and paste the SMILES string to reconstruct a **3D geometry** from the code.

In [ ]:
#@title <small><small> { display-mode: "form" }

# mute warnings/erros in console
RDLogger.logger().setLevel(RDLogger.CRITICAL)

molecule = "NO2"
GEOMINFO = load_n2o4_2no2()[4]
# ------------------------------------------------
if "MOLECULES_ID" not in globals(): MOLECULES_ID  = {k:None for k in MOLECULES}
# ------------------------------------------------
MOLECULES_ID[molecule] = input(rf"insert the SMILES code for {molecule:s} : ").strip()
if MOLECULES_ID[molecule].startswith('"'): MOLECULES_ID[molecule] = MOLECULES_ID[molecule][1:]
if MOLECULES_ID[molecule].endswith('"')  : MOLECULES_ID[molecule] = MOLECULES_ID[molecule][:-1]
print("")

# Get geometry
try:
   mol_id    = MOLECULES_ID[molecule]
   xyz_guess = files_of_interest(molecule)[0]
   print(rf"Retrieving geometry for: {str(mol_id):s}")
   print("")
   symbols,coords,smiles = rdkit_smiles2geom(mol_id)
   data_2_xyz(symbols,coords,xyz_guess,smiles)
   print(rf"     - geometry stored in '{xyz_guess:s}'")
   info = geometric_info_xyz(xyz_guess,GEOMINFO[molecule])
   print(info)
   print("")
   view = create_visualization_xyz(xyz_guess)
   view.show()

except:
   print(rf"ERROR! Something went wrong! Did you introduce the correct SMILES??")
   print("")

#####
The molecular structures obtained so far are **not** optimized; they do not correspond to the **minimum-energy geometries**.

Before performing any optimization, however, we must specify for each molecule both its total charge and the number of unpaired electrons. In our case:

* N$_2$O$_4$: **neutral** molecule with **no** unpaired electrons
* NO$_2$ &nbsp;: **neutral** molecule with **one** unpaired electron

Run the next cell and enter the value for these variables:

In [ ]:
#@title <small><small> { display-mode: "form" }

CHARGES,UNPAIREDS = {},{}
print("Insert the total charge (a.u.) and number of unpaired electrons for each molecule (integers only):")
print("")
for molecule in MOLECULES:
    print(rf"   * {molecule:s}")
    charge   = input("     total molecular charge [in au] = ").strip()
    unpaired = input("     number of unpaired electrons   = ").strip()
    CHARGES[molecule]   = int(charge)
    UNPAIREDS[molecule] = int(unpaired)
    print("")

#### **(b) DFT calculations**

#####

Now, we are ready to perform the quantum-chemistry calculations using **PySCF**[[🌐]](https://pyscf.org). After optimizing the geometries, we will also compute the **Hessian matrix** to verify that each structure corresponds to a true minimum (all vibrational frequencies are real), rather than a different type of stationary point. Moreover, these vibrational frequencies are required to obtain the vibrational partition function, which is essential for evaluating thermodynamic quantities.

We will test the following DFT levels of theory:

* B3LYP-D4
* O3LYP-D4
* PBE0-D4

with the **6-31G*** basis set. Run the next cell to load these methods and the basis set.

<br>

**Note:** In PySCF, the selected basis set corresponds to "**6-31G*** **5d**" in _Gaussian_ electronic structure software [[🌐]](https://gaussian.com/basissets/).

In [ ]:
#@title <small><small> { display-mode: "form" }

# =====================================================
# Classroom setting DFTGRID = 2 (short runtime).
# For higher accuracy, set DFTGRID = 5 (slower).
#------------------------------------------------------
DFTGRID       = 2
# =====================================================

#------------------------------------------------------
FUNCTIONALS   = "b3lyp-d4,o3lyp-d4,pbe0-d4".split(",")
BASIS         = "6-31g*"
#------------------------------------------------------
print(f"The following DFT functionals will be tested in combination with the {BASIS.upper():s} basis set:")
for functional in FUNCTIONALS:
    print(f"   * {functional.upper():s}")
print("")
print(f"Grid level is set to: {DFTGRID:d}")
print("")
#------------------------------------------------------
if "DFTDATA" not in globals(): DFTDATA = {k:{}   for k in MOLECULES}
#------------------------------------------------------

Now run the next three cells in sequence to perform the quantum-chemistry calculations.

 * ⏳ Each cell typically takes **~6 minutes** to run. ⏳

<br>

**Note:** To keep runtimes manageable, we use a coarse DFT integration grid. More accurate results are obtained with a **level 5** grid. You can switch to it by changing `DFTGRID` from **2** to **5** in the previous cell and re-running it; this usually increases the runtime to **~10–15 min per cell**.



In [ ]:
#@title <small>💻 Optimizing guess structures with B3LYP-D4 <small> { display-mode: "form" }

key = (FUNCTIONALS[0],BASIS)
print(rf" - Functional: {key[0].upper():s}")
print(rf" - Basis set : {key[1].upper():s}")
print(rf" - Grid level: {DFTGRID:d}")
print("")

t1 = time.time()
for molecule in MOLECULES:
    print(rf" * Molecule: {molecule:s}")
    DFTDATA[molecule][key] = optimize_and_freqs(molecule,UNPAIREDS[molecule],CHARGES[molecule],key[0],key[1],bsym=True)
    print("")
    pyscf_printdata(DFTDATA[molecule][key])
    pyscf_download(molecule,key[0],key[1],[1])
    print("")
t2 = time.time()
print(" - total elapsed time: %.2f minutes"%((t2-t1)/60))

In [ ]:
#@title <small>💻 Optimizing guess structures with O3LYP-D4 <small> { display-mode: "form" }

key = (FUNCTIONALS[1],BASIS)
print(rf" - Functional: {key[0].upper():s}")
print(rf" - Basis set : {key[1].upper():s}")
print(rf" - Grid level: {DFTGRID:d}")
print("")

t1 = time.time()
for molecule in MOLECULES:
    print(rf" * Molecule: {molecule:s}")
    DFTDATA[molecule][key] = optimize_and_freqs(molecule,UNPAIREDS[molecule],CHARGES[molecule],key[0],key[1],bsym=True)
    print("")
    pyscf_printdata(DFTDATA[molecule][key])
    pyscf_download(molecule,key[0],key[1],[1])
    print("")
t2 = time.time()
print(" - total elapsed time: %.2f minutes"%((t2-t1)/60))

In [ ]:
#@title <small>💻 Optimizing guess structures with PBE0-D4 <small> { display-mode: "form" }

key = (FUNCTIONALS[2],BASIS)
print(rf" - Functional: {key[0].upper():s}")
print(rf" - Basis set : {key[1].upper():s}")
print(rf" - Grid level: {DFTGRID:d}")
print("")

t1 = time.time()
for molecule in MOLECULES:
    print(rf" * Molecule: {molecule:s}")
    DFTDATA[molecule][key] = optimize_and_freqs(molecule,UNPAIREDS[molecule],CHARGES[molecule],key[0],key[1],bsym=True)
    print("")
    pyscf_printdata(DFTDATA[molecule][key])
    pyscf_download(molecule,key[0],key[1],[1])
    print("")
t2 = time.time()
print(" - total elapsed time: %.2f minutes"%((t2-t1)/60))

#####
Run the cell below to **visualize** the optimized geometries. Again, the views are interactive; you can rotate and inspect the molecule freely.

In [ ]:
#@title <small><small> { display-mode: "form" }

GEOMINFO = load_n2o4_2no2()[4]
print("")
for molecule in MOLECULES:

    html_blocks = []
    print(rf"==> molecule: {molecule:s}")
    for functional in FUNCTIONALS:

        xyz_opt = files_of_interest(molecule,functional,BASIS)[1]
        if not os.path.exists(xyz_opt): continue
        # generate visualization of molecule
        view = create_visualization_xyz(xyz_opt)
        # read xyz file and get some interesting geometric features
        info  = functional+"/"+BASIS + "\n"
        info += geometric_info_xyz(xyz_opt,GEOMINFO[molecule])
        # save for later visualization
        html_blocks.append(f"""
            <div style='text-align:center; margin:10px;'>
                {view._make_html()}
                <div style='font-size:16px; font-weight:500; margin-top:-2px;white-space:pre-line;'>
                    {info}
                </div>
            </div>
        """)

    # Displays all viewers side by side
    html_code = "<div style='display:flex; gap:100px; width:100%;'>" + "".join(html_blocks) + "</div>"
    display(HTML(html_code))
    print("")

#####

PySCF does not always return the correct symmetry number ($\sigma$) due to numerical inaccuracies [[🌐]](https://link.springer.com/article/10.1007/s00214-007-0328-0). Therefore, before moving to (c) and compute the partition functions, we must ensure that the symmetry numbers are correct. For the species involved in the reaction studied here, the appropriate values are:

* N$_2$O$_4$ ==> D2h symmetry ==> $\sigma = 4$
* NO$_2$ &nbsp;  ==> C2v symmetry ==> $\sigma = 2$

Verify these values in the preceding optimization cells, and run the following cell if any symmetry number needs to be corrected.

In [ ]:
#@title <small>💻 Correct symmetry numbers <small> { display-mode: "form" }
print("Insert the rotational symmetry number for each molecule:")
print("")
for molecule in MOLECULES:
    sigma = int(input(rf"   * sigma({molecule:s}) = ").strip())
    print("")
    for functional in FUNCTIONALS:
        key = (functional,BASIS)
        try:
          sigma_old = DFTDATA[molecule][key]["rotsigma"]
          print(rf"     {functional.upper():8s}: {sigma_old:d} --> {sigma:d}")
          DFTDATA[molecule][key]["rotsigma"] = sigma
        except:
          print(rf"     data not found for {functional.upper():s}...")
    print("")

#### **(c) Calculation of partition functions and thermodynamic functions**


#####
We can now evaluate the partition functions of all species participating in the reaction at 298.15 K. From them, the thermodynamic state functions can be determined.

In [ ]:
#@title <small><small> { display-mode: "form" }

TABLE  = " ----------------------------------------------------------------------------------------------" + "\n"
TABLE += "  FUNCTIONAL/BASIS_SET | DELTA_r{U}^o | DELTA_r{H}^o | DELTA_r{S}^o | DELTA_r{G}^o |   K_p^o   " + "\n"
TABLE += " ----------------------|--------------|--------------|--------------|--------------|-----------" + "\n"

print("Partition functions and zero-point–corrected total energies (E0 + ZPE); energies in hartree.")
print("")

TREF          = load_n2o4_2no2()[0][3]
print(f"    * Reference temperature: {TREF:.2f} K\n")
DFT_REFTHERMO = {}
for functional in FUNCTIONALS:
    key   = (functional,BASIS)
    level = rf"{key[0].upper():8s}/{key[1].upper():s}"
    allok = True

    SUBTABLE  = "    --------------------------------------------------------------------------------------------"+"\n"
    SUBTABLE += "      molecule |   pfn tr    |  pfn rot   |  pfn vib   |  pfn ele   |  pfn tot   |   E0 + ZPE   "+"\n"
    SUBTABLE += "    -----------|-------------|------------|------------|------------|---------------------------"+"\n"

    DUo,DHo,DSo,DGo  = 0.,0.,0.,0.
    for nu_i,molecule in zip(NUS,MOLECULES):

        output_frq = files_of_interest(molecule,key[0],key[1])[3]
        if not os.path.exists(output_frq):
           allok = False
           break
        # calculate thermodynamics
        try: Uo, Ho, So, Go, line = compute_thermodynamics(TREF,molecule,key,DFTDATA)
        except: allok = False; break
        SUBTABLE += rf"      {molecule:8s} | " + line + "\n"

        # reaction magnitudes
        DUo += nu_i * Uo
        DHo += nu_i * Ho
        DSo += nu_i * So
        DGo += nu_i * Go
    SUBTABLE += "    --------------------------------------------------------------------------------------------"+"\n"
    if not allok: continue
    print(rf"    ==> {level:s} <==")
    print("")
    print(SUBTABLE)
    # Equilibrium constant
    Kp = np.exp(-DGo/k_B/TREF)
    # Print info
    if Kp > 1E4: ff = "9.2E"
    else       : ff = "9.3f"
    TABLE += rf"  {level:12s}      | {DUo*NA/1000:12.2f} | {DHo*NA/1000:12.2f} | {DSo*NA:12.2f} | {DGo*NA/1000:12.2f} | {Kp:{ff:s}} " + "\n"
    # save reaction magnitudes
    DFT_REFTHERMO[key] = TREF,DUo*NA,DHo*NA,DSo*NA,DGo*NA
TABLE += " ----------------------------------------------------------------------------------------------" + "\n"
TABLE += "\n"
TABLE += "     units of DELTA_r{U}^o ==> kJ/mol"   + "\n"
TABLE += "     units of DELTA_r{H}^o ==> kJ/mol"   + "\n"
TABLE += "     units of DELTA_r{S}^o ==>  J/mol/K" + "\n"
TABLE += "     units of DELTA_r{G}^o ==> kJ/mol"   + "\n"

print("")
print(TABLE)
print("")

#####

The experimental value for $\Delta_{r}G^\circ$(298.15 K) is 4.74 kJ/mol, and O3LYP-D4/6-31G* provides the closest agreement among the tested levels of theory. Therefore, we will use this functional for the final step (a.3).

#### **(d) Temperature dependence of $\Delta_{r}G^\circ$: $\Delta_{r}C_{p}^\circ$ and the vibrational degrees of freedom**

#####

The calculation of $\Delta_{r}G^\circ$ in (c) can be performed at temperatures other than 298.15 K. Let us check how the O3LYP-D4/6-31G* level predicts $\Delta_{r}G^\circ$ over the 200-400 K temperature range.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
key  = (FUNCTIONALS[1],BASIS)
#------------------------------------------------------------------------------#
Tmin = 200
Tmax = 400
#------------------------------------------------------------------------------#
T   = np.linspace(Tmin,Tmax,num=51)
#------------------------------------------------------------------------------#
# Calculate D_{r}G^o at different temperatures using Statistical Thermodynamics
print(rf"Computing Delta_{{r}}G^o from {T[0]:.2f} to {T[-1]:.2f} K using Statistical Mechanics...")
print("")
print(rf" - Functional: {key[0].upper():s}")
print(rf" - Basis set : {key[1].upper():s}")
print("")
DGo_T = []
for Ti in list(T):
    DGo_i = [nu_i * compute_thermodynamics(Ti,molecule,key,DFTDATA)[3] for nu_i,molecule in zip(NUS,MOLECULES)]
    DGo_i = float(sum(DGo_i)*NA)
    DGo_T.append(DGo_i)
DGo_T = np.array(DGo_T)

# Data at reference temperature
TREF,DUo_qc,DHo_qc,DSo_qc,DGo_qc = DFT_REFTHERMO[key]

# Plot data
plot_DGo_T_statmech(T,DGo_T,None,None,TREF,DGo_qc,key)

#####

As you may recall from the first part of this Notebook, this temperature dependence can be written as:

$$
\Delta_{r} G^{\circ}(T) = \Delta_{r} H^{\circ}(T_{\rm ref})-T \cdot \Delta_{r} S^{\circ}(T_{\rm ref})+\Delta_{r} C_p^\circ \cdot \left[ T-T_{\rm ref}+T \cdot \ln \left(\frac{T_{\rm ref}}{T}\right)\right]
\tag{1}
$$

under the assumption that $\Delta_{r}C_{p}^\circ$ is constant. Note that we already calculated all quantities at the reference temperature ($T_{\rm ref}$ = 298.15 K), except for $\Delta_{r}C_{p}^\circ$ itself.

Next, we will explore how the choice of $\Delta_{r}C_{p}^\circ$ influences the predicted temperature dependence of $\Delta_{r} G^{\circ}(T)$, and which value best reproduces the numerical results obtained in the previous cell. Run the following cell as many times as you want, enter a trial value of $\Delta_{r}C_{p}^\circ$, and compare the curve from Eq. (1) with the previously computed one.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
key = (FUNCTIONALS[1],BASIS)
#------------------------------------------------------------------------------#
Tmin = 200
Tmax = 400
#------------------------------------------------------------------------------#
T    = np.linspace(Tmin,Tmax,num=51)
# Data at reference temperature
TREF,DUo_qc,DHo_qc,DSo_qc,DGo_qc = DFT_REFTHERMO[key]
#------------------------------------------------------------------------------#

# Values at optimized results (and other values)
print(" We can see how data fit according to the value of Delta_{r}C_{p}^o = constant * R")
DCP = float(input("   - insert value for the constant: ").strip())

DGo_model = get_DGo(T,(DHo_qc,DSo_qc,DCP*R,TREF))
RMS       = calculate_rms(DGo_T,DGo_model)/1000
print("   * Delta_{r}C_{p}^o = %5.2f*R ==> RMS = %.3f kJ/mol"%(DCP,RMS))
print("")
plot_DGo_T_statmech(T,DGo_T,DGo_model,DCP*R,TREF,DGo_qc,key)
print()

#####

The optimal value of $\Delta_{r} C_p^\circ$ can be obtained directly by fitting the computed $\Delta_{r} G^\circ(T)$ values (calculated using Statistical Thermodynamics) to Eq. (1). In other words, instead of choosing  $\Delta_{r} C_p^\circ$ manually, one could determine the value that minimizes the deviation between the analytical expression, Eq. (1), and the data, yielding the best agreement across the temperature range.

To do this, run the following cell.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
key = (FUNCTIONALS[1],BASIS)
#------------------------------------------------------------------------------#
Tmin = 200
Tmax = 400
#------------------------------------------------------------------------------#
T   = np.linspace(Tmin,Tmax,num=51)
# Data at reference temperature
TREF,DUo_qc,DHo_qc,DSo_qc,DGo_qc = DFT_REFTHERMO[key]
#------------------------------------------------------------------------------#

# minimize
print("Obtaining best value for Delta_{r}C_{p}^o by least squares...")
print("")
res       = minimize_scalar(sum_squared_errors,bounds=(-5,5),method='bounded',args=(T,DGo_T,TREF,DHo_qc,DSo_qc))
DCP_best  = res.x

# Values at optimized results (and other values)
DGo_model = get_DGo(T,(DHo_qc,DSo_qc,DCP_best*R,TREF))
RMS       = calculate_rms(DGo_T,DGo_model)/1000

print("   * best fit --> Delta_{r}C_{p}^o = %5.2f*R   [RMS = %.3f kJ/mol]"%(DCP_best,RMS))
print("")

# Plotting
plot_DGo_T_statmech(T,DGo_T,DGo_model,DCP_best*R,TREF,DGo_qc,key)
print()

#####
The optimal value of $\Delta_{r} C_p^\circ$ can also be estimated from statistical mechanics. In this approach, the molar heat capacity at constant pressure of a given molecule is written as:

$$
C_{p,m} = R + \frac{3 + n^{R^\ast} + 2 \cdot n^{V^\ast}}{2} \cdot R
\tag{8}
$$

where $n^{R^\ast}$ and $n^{V^\ast}$ are the effective rotational and vibrational degrees of freedom (dofs), respectively. The term 3 accounts for the traslational dofs. For linear molecules, $n^{R^\ast}=2$; otherwise, $n^{R^\ast}=3$.

The vibrational contribution, $n^{V^\ast}$, is obtained by summing the contributions from each normal mode (frequency $\nu_j$):

$$
n^{V^\ast} = \sum_j n^{V^\ast}_j
\tag{9}
$$

where each mode contributes:

$$
n^{V^\ast}_j = \left( \frac{\theta_j^V}{T} \cdot \frac{\exp(-\theta_j^V/2T)}{1-\exp(-\theta_j^V/T)} \right)^2
\tag{10}
$$

In the previous equation, $\theta_j^V = h \nu_j / k_B$, where $h$ and $k_B$ are the Planck and the Boltzmann constants, respectively.

To explore how $n^{V^\ast}_j$ depends on the vibrational frequency, $\nu_j$, run the following cell.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
key = (FUNCTIONALS[1],BASIS)
#------------------------------------------------------------------------------#
Tmin = 200
Tmax = 400
#------------------------------------------------------------------------------#

# Find limits for the frequencies
freqmin = +float("inf")
freqmax = -float("inf")
for molecule in DFTDATA.keys():
    freqs = DFTDATA[molecule][key]["freqs"]
    freqmin = min(freqmin,min(freqs))
    freqmax = max(freqmax,max(freqs))
freqmin_cm = np.floor(freqmin/100)
freqmax_cm = np.floor(freqmax/100)

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args        = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
freq_slider = w.FloatSlider(value=freqmin_cm,min=freqmin_cm,max=freqmax_cm,step=0.01,description=r'freq [1/cm]', **args)
ui       = w.VBox([freq_slider])

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: _on_download_clicked(b,"dof"))

# -------- slider ---> function --------
# notice that P is converted from bar to Pa with *1E5
out = w.interactive_output(lambda freq: plot_vib_average(Tmin,Tmax,freqmin_cm,freqmax_cm,freq), {'freq': freq_slider})
display(w.VBox([ui, btn]), out)

#####

Run the following cell to compute $C_{p,m}$ for each molecule involved in the reaction, as well as $\Delta_{r} C_p^\circ$. You will see that the value obtained is the same as the optimal value.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
key = (FUNCTIONALS[1],BASIS)
#------------------------------------------------------------------------------#
Tmin = 200
Tmax = 400
#------------------------------------------------------------------------------#

print("-------------------------------------------")
print(" Molecule | rot dof  | vib dof  |  C_{p,m} ")
print("-------------------------------------------")
DCP = 0.0
for i,molecule in enumerate(MOLECULES):
    # check if data is available
    if molecule not in DFTDATA: continue
    if key      not in DFTDATA[molecule]: continue
    # rotational dofs
    if DFTDATA[molecule][key]["islinear"]: rdof = 2
    else                                 : rdof = 3
    # calculate vdof and Cp
    freqs_cm = [ii/100 for ii in DFTDATA[molecule][key]["freqs"]]
    vdof     = sum([vib_contri_avera(freq_cm,Tmin,Tmax) for freq_cm in freqs_cm])
    Cp       = R + (3+rdof+2*vdof)/2*R
    DCP     += Cp*NUS[i]
    print(rf" {molecule:7s}  |  {rdof:6d}  |  {vdof:6.3f}  | {Cp/R:6.3f}*R  ")
print("-------------------------------------------")
print(rf"             ==> Delta_r{{C_p}}^o = {DCP/R:6.2f}*R")
print("")

### **3. A Chemical Kinetics Perspective**  


####
In this section, we will analyze the connection between **chemical equilibrium** and **chemical kinetics**.

Our goal is to explore how equilibrium is reached as a _dynamic process_. We will consider the two classical scenarios: constant-_T,p_ and constant-_T,V_.

####
**(a) Kinetic background**


#####
To connect kinetics with thermodynamics, we require the forward ($k_{fw}$) and backward ($k_{bw}$) rate constants. For the reaction here studied, these rate constants are related to the equilibrium constant according to:

$$
K_p^\circ(T) \cdot \frac{p^\circ}{RT} = \frac{k_{fw}}{k_{bw}}
 \tag{11}
$$

where $p^\circ = 1$ bar is the standard pressure. Thus, if the temperature dependence of $k_{fw}(T)$ is known, the backward rate constant follows immediately.

For the forward reaction, we can adopt an Arrhenius-type expression:

$$
k_{fw} = A \cdot e^{-B/T}
 \tag{12}
$$

where $A$ is the pre-exponential factor and $B$ is related to the activation energy. In particular, we will use the parametrization reported in _Atmos. Chem. Phys._ **4**, 1461-1738 (2004)[[🌐]](https://doi.org/10.5194/acp-4-1461-2004):

$$
k_{fw} = 1.15 \cdot 10^{16}  \cdot e^{-6460/T} \;\;\; s^{-1}
 \tag{13}
$$

Although this expression is formally valid only in the 250–300 K range and corresponds to the high-pressure limit, we will apply it over the full temperature interval for pedagogical purposes.

<br>

Execute the following cell to load the values of $A$ and $B$.
[_You may adjust any parameter in the cell prior to execution if you wish to explore alternative data or hypothetical scenarios._]

In [ ]:
#@title <small> <small> { display-mode: "form" }
arrhenius_A = 1.15E16 # in 1/s
arrhenius_B = 6460    # in K
print(rf"Value for A: {arrhenius_A} 1/s")
print(rf"Value for B: {arrhenius_B} K")

#### **(b) Carrying out the experiments**

#####
Use the following cell to configure and analyze a scenario by selecting the initial temperature, pressure, and volume of the system. You may also specify the initial mole fraction of the reactant (N$_2$O$_4$) and choose whether the experiment is performed under constant-_T,p_ or constant-_T,V_ conditions.
The plots will update dynamically as you adjust the initial conditions using the sliders.

In [ ]:
#@title <small> <small> { display-mode: "form" }

T_ref = 298.00
Tmin  = 200.00
Tmax  = 400.00

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args      = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
T_slider  = w.FloatSlider(value=T_ref,min=Tmin,max=  Tmax,step=1.00,description=r'T [K]'  , **args)
P_slider  = w.FloatSlider(value= 1.00,min=0.01,max=  3.00,step=0.01,description=r'p [bar]', **args)
V_slider  = w.FloatSlider(value=24.78,min=2.00,max=250.00,step=0.01,description=r'V [L]'  , **args)
yA_slider = w.FloatSlider(value= 1.00,min=0.00,max=  1.00,step=0.01,description=r'y(N2O4)', **args)

# -------- Scenario selector --------
selector = w.RadioButtons(
    options=[('constant-T,V', 'kinVT'),
             ('constant-T,p', 'kinPT')],
    style={'description_width': '100px'},
    layout=w.Layout(width='260px', margin='0 0 0 40px')
)

# -------- download button --------
btn = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: _on_download_clicked(b,selector.value))

# -------- slider ---> function --------
# P*1E+5 : bar --> Pa
# V*1E-3 : L   --> m3
string = w.Output()
out    = w.interactive_output(lambda T0, P0, V0, yA0, scenario: kinetics_N2O4(T0, P0*1E5, V0*1E-3,yA0,scenario), {'T0': T_slider, 'P0': P_slider,'V0': V_slider,'yA0': yA_slider,'scenario':selector})

# Print sliders and selector
print("Set the initial conditions:")
ui = w.VBox([T_slider,P_slider,V_slider,yA_slider])
display(ui)
print("Select scenario:")
display(w.VBox([selector]))

# Print download button and plot
display(w.VBox([btn,out]))

#####

By running the next cell, the information about both the initial state and the equilibrium state corresponding to the most recent experiment configured in the previous cell will be displayed.

In [ ]:
#@title <small> <small> { display-mode: "form" }
print(last_info)